In [1]:
# ============================================================
# CATHEDRAL FULL PIPELINE
# ============================================================
# This single script combines your three cathedral notebooks:
#   1) Data_Frame_Creation_Cathedral.ipynb
#   2) Cathedral_with_style.ipynb
#   3) Cathdral_Filter.ipynb
#
# It preserves the same source priority and transformation logic:
#   raw openICPSR files -> cathedral_master.csv -> Wikidata/Wikipedia enrichment
#   -> cathedral_master_with_wikidata_styles_with_periods.csv -> print final columns.
#
# Before running, edit PROJECT_DIR below if your folder location changes.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re

# =========================
# 0) USER CONFIG
# =========================
# Folder that contains the openICPSR cathedral files.
# Expected files inside DAT: statobs.csv, heights.csv, dynobs.csv, disasters.csv, italyheights.csv
# Expected Excel file inside EXCELS: saints and exogenous disasters.xlsx
PROJECT_DIR = Path("/Users/jeeveshattri/Desktop/Monuments_Of_Power/Background_Data")
REPO_DIR = PROJECT_DIR / "cathedrals"
DAT = REPO_DIR / "dat"
EXCELS = REPO_DIR / "excels"

# Outputs. These names preserve your old intermediate/final file names.
OUT_DIR = PROJECT_DIR
BASE_OUTPUT_CSV = OUT_DIR / "cathedral_master.csv"
FINAL_OUTPUT_CSV = OUT_DIR / "cathedral_master_with_wikidata_styles_with_periods.csv"
OUT_PATH = BASE_OUTPUT_CSV

# Preserve the old notebook behavior by default.
# If set to True, the final file recomputes duration/period_bucket/construction-window disasters
# after Wikipedia date reconciliation. Leave False for exact old-output compatibility.
RECOMPUTE_DERIVED_AFTER_WIKI_DATE_UPDATES = False

# =========================
# Helpers
# =========================
def nzmin(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    return float(s.min()) if len(s) else np.nan

def nzmax(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    return float(s.max()) if len(s) else np.nan

def clean_key(x):
    if pd.isna(x):
        return ""
    x = str(x).lower()
    x = x.replace("’", "").replace("'", "")
    x = re.sub(r"[^a-z0-9\s]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def normalize_osmid(series) -> pd.Series:
    """
    FORCE every osmid to float with 1 decimal, everywhere.
    This makes 100010435 and 100010435.0 match as 100010435.0.
    """
    s = pd.to_numeric(series, errors="coerce")
    # round to 1 decimal to be explicit (though these are integers)
    s = s.round(1)
    return s.astype("float64")

def read_csv_safely(path, **kwargs):
    """
    Robust CSV reader:
    - tries utf-8 first
    - then cp1252
    - then latin1
    - finally utf-8 with replacement
    """
    for enc in ["utf-8", "cp1252", "latin1"]:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except UnicodeDecodeError:
            pass
    return pd.read_csv(path, encoding="utf-8", encoding_errors="replace", **kwargs)

def coalesce(*cols):
    out = None
    for c in cols:
        if c is None:
            continue
        if out is None:
            out = c.copy()
        else:
            out = out.where(~out.isna(), c)
    return out

def truthy_series(col: pd.Series) -> pd.Series:
    return col.astype(str).str.strip().str.lower().isin(["true", "1", "t", "yes", "y"])

def collapse_series(s: pd.Series):
    vals = pd.unique(s.dropna().astype(str).map(lambda x: x.strip()).replace({"": np.nan}).dropna())
    return ";".join(vals) if len(vals) else np.nan

# =========================
# 1) Base cathedral list (source of truth)
# =========================
stat_path = DAT / "statobs.csv"
statobs = read_csv_safely(stat_path, low_memory=False)
statobs.columns = [c.strip().lower() for c in statobs.columns]

required = {"osmid", "category"}
missing = required - set(statobs.columns)
if missing:
    raise ValueError(f"statobs.csv missing required columns: {missing}")

statobs["category"] = statobs["category"].astype(str).str.lower()
statobs["osmid"] = normalize_osmid(statobs["osmid"])

cathedrals = (
    statobs.loc[statobs["category"].eq("cathedral")]
    .dropna(subset=["osmid"])
    .drop_duplicates(subset=["osmid"])
    .copy()
)

# =========================
# 2) Heights (best static start/finish/size)
# =========================
heights_path = DAT / "heights.csv"
heights = None
if heights_path.exists():
    heights = read_csv_safely(heights_path, low_memory=False)
    heights.columns = [c.strip().lower() for c in heights.columns]
    if "osmid" in heights.columns:
        heights["osmid"] = normalize_osmid(heights["osmid"])

    heights = heights.rename(columns={
        "church": "height_name",
        "city": "height_city",
        "ctr": "ctr_h",
        "ground_surface_m2": "surface_m2_h",
        "height": "height_m_h",
        "start": "start_h",
        "finish": "end_h",
        "remarks": "height_remarks",
    })

# =========================
# 3) dynobs panel -> per-cathedral aggregates
#    NOTE: dynobs is ONLY used for start_dyn/end_dyn (nothing else)
# =========================
dyn_path = DAT / "dynobs.csv"
dyn_agg = None
if dyn_path.exists():
    dyn = read_csv_safely(dyn_path, low_memory=False)
    dyn.columns = [c.strip().lower() for c in dyn.columns]
    if "osmid" in dyn.columns:
        dyn["osmid"] = normalize_osmid(dyn["osmid"])
    dyn["year"] = pd.to_numeric(dyn.get("year"), errors="coerce")

    def start_year_from_newbld(g):
        if "newbld" in g.columns:
            nb = truthy_series(g["newbld"])
            if nb.any():
                return float(g.loc[nb, "year"].min())
        return float(g["year"].min())

    dyn_agg = (
        dyn.dropna(subset=["osmid"])
           .groupby("osmid", as_index=False)
           .apply(lambda g: pd.Series({
               "start_dyn": start_year_from_newbld(g),
               "end_dyn": float(g["year"].max()),
           }))
           .reset_index(drop=True)
    )
    dyn_agg["osmid"] = normalize_osmid(dyn_agg["osmid"])

# =========================
# 4) Disasters -> per-cathedral event summary
# =========================
dis_path = DAT / "disasters.csv"
dis_agg = None
window = None

if dis_path.exists():
    dis = read_csv_safely(dis_path, low_memory=False)
    dis.columns = [c.strip().lower() for c in dis.columns]
    if "osmid" in dis.columns:
        dis["osmid"] = normalize_osmid(dis["osmid"])
    dis["year"] = pd.to_numeric(dis.get("year"), errors="coerce")
    dis["cause"] = dis.get("cause", "").astype(str)

    dis_agg = (
        dis.dropna(subset=["osmid"])
           .groupby("osmid", as_index=False)
           .apply(lambda g: pd.Series({
               "n_disasters": len(g),
               "any_war": bool(truthy_series(g["war"]).any()) if "war" in g.columns else False,
               "any_fire": bool(truthy_series(g["fire"]).any()) if "fire" in g.columns else False,
               "any_earthquake": bool(truthy_series(g["earthquake"]).any()) if "earthquake" in g.columns else False,
               "any_other_natural": bool(truthy_series(g["other_natural"]).any()) if "other_natural" in g.columns else False,
               "any_other": bool(truthy_series(g["other"]).any()) if "other" in g.columns else False,
               "any_collapse_text": bool(g["cause"].str.lower().str.contains("collaps", na=False).any()),
               "first_disaster_year": nzmin(g["year"]),
               "last_disaster_year": nzmax(g["year"]),
               "disaster_years": ";".join(map(str, sorted(g["year"].dropna().unique()))),
           }))
           .reset_index(drop=True)
    )
    dis_agg["osmid"] = normalize_osmid(dis_agg["osmid"])

# =========================
# 5) Saints/Destroyed Excel (ROW3-wins header + exact date pairing)
#    Output: saints_agg with ONLY [osmid, surface, saints_compiled, damage_compiled]
# =========================
saints_path = EXCELS / "saints and exogenous disasters.xlsx"
saints_agg = None

def norm_header(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    return re.sub(r"\s+", " ", str(x).strip())

def make_unique(cols):
    seen = {}
    out = []
    for c in cols:
        c = norm_header(c)
        if c in seen:
            seen[c] += 1
            out.append(f"{c}__{seen[c]}")
        else:
            seen[c] = 0
            out.append(c)
    return out

def _is_blank(x) -> bool:
    if x is None:
        return True
    if isinstance(x, float) and np.isnan(x):
        return True
    s = str(x).strip()
    return s == "" or s.lower() in {"nan", "none", "no info", "n/a"}

def _fmt_date(x) -> str:
    return "N/A" if _is_blank(x) else str(x).strip()

def _split_items(v):
    items = re.split(r"[;,/]| and ", str(v), flags=re.IGNORECASE)
    return [t.strip() for t in items if t and t.strip()]

def _dedupe_comma_string(val):
    if pd.isna(val):
        return np.nan
    parts = [p.strip() for p in str(val).split(",") if p.strip()]
    seen = set()
    out = []
    for p in parts:
        key = re.sub(r"\s+", "", p).lower()  # ignore spaces + case
        if key not in seen:
            seen.add(key)
            out.append(p)
    return ", ".join(out) if out else np.nan

def compile_pairs(df: pd.DataFrame, pairs):
    cols_set = set(df.columns)
    pairs = [(v, d) for (v, d) in pairs if v in cols_set]  # keep only existing

    def build_row(row):
        pieces = []
        for vcol, dcol in pairs:
            v = row.get(vcol, np.nan)
            if _is_blank(v):
                continue
            d = row.get(dcol, np.nan) if dcol in cols_set else np.nan
            dtxt = _fmt_date(d)
            for item in _split_items(v):
                pieces.append(f"{item} ({dtxt})")
        return ", ".join(pieces) if pieces else np.nan

    return df.apply(build_row, axis=1)

def read_excel_merge_row2_row3_row3_wins(path: Path) -> pd.DataFrame:
    """
    Ignore row1.
    Header per column = row3 if present else row2; drop columns where both blank.
    Data starts at row4 (0-based index 3).
    Concatenate all sheets; keep `sheet` for debugging.
    """
    xls = pd.ExcelFile(path, engine="openpyxl")
    frames = []

    for sh in xls.sheet_names:
        raw = pd.read_excel(path, sheet_name=sh, engine="openpyxl", header=None)
        if raw.shape[0] < 4:
            continue

        h2 = raw.iloc[1].tolist()
        h3 = raw.iloc[2].tolist()

        headers, keep_idx = [], []
        for j, (a, b) in enumerate(zip(h2, h3)):
            a = norm_header(a)
            b = norm_header(b)
            name = b if b != "" else a
            if name == "":
                continue
            keep_idx.append(j)
            headers.append(name)

        df = raw.iloc[3:, keep_idx].copy()
        df.columns = make_unique(headers)
        df = df.dropna(how="all")
        df.insert(0, "sheet", sh)
        frames.append(df)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

if saints_path.exists():
    raw_df = read_excel_merge_row2_row3_row3_wins(saints_path)

    # normalize osmid if present
    if "osmid" in raw_df.columns:
        raw_df["osmid"] = normalize_osmid(raw_df["osmid"])

    # --- exact mapping you specified ---
    saints_pairs = [
        ("saints", "(approximate) date"),
        ("saints__1", "(approximate) date__1"),
        ("saints__2", "(approximate) date__2"),
        ("saints__3", "(approximate) date__3"),
    ]
    destroyed_pairs = [
        ("destroyed", "(approximate) date__4"),
        ("destroyed__1", "(approximate) date__5"),
        ("destroyed__2", "(approximate) date__6"),
    ]

    raw_df["saints_compiled"] = compile_pairs(raw_df, saints_pairs).apply(_dedupe_comma_string)
    raw_df["damage_compiled"] = compile_pairs(raw_df, destroyed_pairs).apply(_dedupe_comma_string)

    # surface column name in your sheet is "surface" after row3-wins (keep as-is)
    # If it sometimes appears as "surface__1" etc., coalesce it here.
    surface_col = "surface" if "surface" in raw_df.columns else ("surface__1" if "surface__1" in raw_df.columns else None)

    saints_agg = pd.DataFrame({
        "osmid": raw_df["osmid"] if "osmid" in raw_df.columns else np.nan,
        "surface": raw_df[surface_col] if surface_col else np.nan,
        "saints_compiled": raw_df["saints_compiled"],
        "damage_compiled": raw_df["damage_compiled"],
    })

    # keep only rows with osmid (since this will be merged later on osmid)
    saints_agg = saints_agg.dropna(subset=["osmid"]).copy()
    saints_agg["osmid"] = normalize_osmid(saints_agg["osmid"])

    # if the same osmid appears multiple times across sheets/rows, collapse to one row per osmid
    def collapse_unique_series(s: pd.Series):
        vals = [v.strip() for v in s.dropna().astype(str) if v.strip()]
        if not vals:
            return np.nan
        # de-dupe ignoring whitespace + case while preserving order
        seen = set()
        out = []
        for v in vals:
            k = re.sub(r"\s+", "", v).lower()
            if k not in seen:
                seen.add(k)
                out.append(v)
        return ", ".join(out)

    saints_agg = (
        saints_agg.groupby("osmid", as_index=False)
                  .agg(
                      surface=("surface", "first"),
                      saints_compiled=("saints_compiled", collapse_unique_series),
                      damage_compiled=("damage_compiled", collapse_unique_series),
                  )
    )
else:
    saints_agg = None

# =========================
# 6) Optional italyheights supplement (no osmid -> fuzzy join)
# =========================
it_path = DAT / "italyheights.csv"
italyheights = None

if it_path.exists():
    it = read_csv_safely(it_path, low_memory=False)
    it.columns = [c.strip().lower() for c in it.columns]

    # normalize facade column name
    if "façade height" in it.columns:
        it = it.rename(columns={"façade height": "facade_height"})
    if "facade height" in it.columns:
        it = it.rename(columns={"facade height": "facade_height"})

    it["it_city_key"] = it.get("city", "").map(clean_key)
    it["it_name_key"] = it.get("name", "").map(clean_key)

    italyheights = it.rename(columns={
        "started": "start_it",
        "finished": "end_it",
        "surface": "surface_m2_it",
        "facade_height": "height_m_it",
    })[["it_city_key", "it_name_key", "start_it", "end_it", "surface_m2_it", "height_m_it"]].copy()

# =========================
# 7) Merge everything into ONE cathedral-per-row table
# =========================
master = cathedrals.copy()
master["osmid"] = normalize_osmid(master["osmid"])

if heights is not None and "osmid" in heights.columns:
    heights["osmid"] = normalize_osmid(heights["osmid"])
    master = master.merge(heights, on="osmid", how="left")

if dyn_agg is not None:
    dyn_agg["osmid"] = normalize_osmid(dyn_agg["osmid"])
    master = master.merge(dyn_agg, on="osmid", how="left")

if dis_agg is not None:
    dis_agg["osmid"] = normalize_osmid(dis_agg["osmid"])
    master = master.merge(dis_agg, on="osmid", how="left")

if saints_agg is not None:
    saints_agg["osmid"] = normalize_osmid(saints_agg["osmid"])
    master = master.merge(saints_agg, on="osmid", how="left")

# italyheights fuzzy join
if italyheights is not None:
    master["city_key"] = master.get("city", "").map(clean_key)
    master["name_key"] = master.get("osmname", "").map(clean_key)
    master = master.merge(
        italyheights,
        left_on=["city_key", "name_key"],
        right_on=["it_city_key", "it_name_key"],
        how="left"
    )

# =========================
# 8) Canonical fields
# =========================
master["start_year"] = coalesce(
    pd.to_numeric(master.get("start_h"), errors="coerce") if "start_h" in master else None,
    pd.to_numeric(master.get("start_it"), errors="coerce") if "start_it" in master else None,
    pd.to_numeric(master.get("start_dyn"), errors="coerce") if "start_dyn" in master else None,
)
master["end_year"] = coalesce(
    pd.to_numeric(master.get("end_h"), errors="coerce") if "end_h" in master else None,
    pd.to_numeric(master.get("end_it"), errors="coerce") if "end_it" in master else None,
    pd.to_numeric(master.get("end_dyn"), errors="coerce") if "end_dyn" in master else None,
)

# dynobs REMOVED from surface/height/m3
master["surface_m2"] = coalesce(
    pd.to_numeric(master.get("surface_m2_h"), errors="coerce") if "surface_m2_h" in master else None,
    pd.to_numeric(master.get("surface_m2_it"), errors="coerce") if "surface_m2_it" in master else None,
)
master["height_m"] = coalesce(
    pd.to_numeric(master.get("height_m_h"), errors="coerce") if "height_m_h" in master else None,
    pd.to_numeric(master.get("height_m_it"), errors="coerce") if "height_m_it" in master else None,
)

master["volume_proxy"] = master["surface_m2"] * master["height_m"]
master["m3"] = master["volume_proxy"]

master["duration_years"] = master["end_year"] - master["start_year"]
master["completed"] = master["end_year"].notna()
master["not_collapsed"] = ~master.get("any_collapse_text", False).fillna(False)

def bucket(y):
    if pd.isna(y):
        return np.nan
    y = float(y)
    if y < 900: return "700–899"
    if y < 1100: return "900–1099"
    if y < 1300: return "1100–1299"
    if y <= 1500: return "1300–1500"
    return "post-1500"

master["period_bucket"] = master["start_year"].map(bucket)

# (Optional but safe): prevent accidental downstream use if these exist
master = master.drop(columns=["surface_m2_dyn", "height_m_dyn", "m3_dyn"], errors="ignore")

# =========================
# 9) Disasters during construction window (counts)
# =========================
if dis_path.exists():
    dis2 = read_csv_safely(dis_path, low_memory=False)
    dis2.columns = [c.strip().lower() for c in dis2.columns]
    dis2["osmid"] = normalize_osmid(dis2["osmid"])
    dis2["year"] = pd.to_numeric(dis2.get("year"), errors="coerce")

    tmp = master[["osmid", "start_year", "end_year"]].merge(dis2, on="osmid", how="left")

    tmp["in_window"] = (
        tmp["year"].notna() & tmp["start_year"].notna() & tmp["end_year"].notna() &
        (tmp["year"] >= tmp["start_year"]) & (tmp["year"] <= tmp["end_year"])
    ).astype(int)

    def boolcol(df, name):
        if name not in df.columns:
            return pd.Series(False, index=df.index)
        return truthy_series(df[name])

    war_b = boolcol(tmp, "war")
    fire_b = boolcol(tmp, "fire")
    eq_b = boolcol(tmp, "earthquake")

    tmp["war_in_window"] = ((tmp["in_window"] == 1) & war_b).astype(int)
    tmp["fire_in_window"] = ((tmp["in_window"] == 1) & fire_b).astype(int)
    tmp["eq_in_window"] = ((tmp["in_window"] == 1) & eq_b).astype(int)

    window = (
        tmp.groupby("osmid", as_index=False)
           .agg(
               disasters_during_construction=("in_window", "sum"),
               war_during_construction=("war_in_window", "sum"),
               fire_during_construction=("fire_in_window", "sum"),
               earthquake_during_construction=("eq_in_window", "sum"),
           )
    )
    window["osmid"] = normalize_osmid(window["osmid"])
    master = master.merge(window, on="osmid", how="left")

# =========================
# 10) Final output columns
# =========================
def col(name, default=np.nan):
    return master[name] if name in master.columns else default

out = pd.DataFrame({
    "osmid": normalize_osmid(master["osmid"]),  # keep forced float .0
    "cathedral_name": col("osmname"),
    "city": col("city"),
    "country": col("ctr"),
    "lat": col("lat"),
    "lon": col("lon"),

    "start_year": master["start_year"],
    "end_year": master["end_year"],
    "duration_years": master["duration_years"],
    "period_bucket": master["period_bucket"],

    "surface_m2": master["surface_m2"],
    "height_m": master["height_m"],
    "m3": master["m3"],
    "volume_proxy": master["volume_proxy"],

    "completed": master["completed"],
    "not_collapsed": master["not_collapsed"],

    "n_disasters": col("n_disasters"),
    "any_war": col("any_war"),
    "any_fire": col("any_fire"),
    "any_earthquake": col("any_earthquake"),
    "any_other_natural": col("any_other_natural"),
    "any_other": col("any_other"),
    "first_disaster_year": col("first_disaster_year"),
    "last_disaster_year": col("last_disaster_year"),
    "disasters_during_construction": col("disasters_during_construction"),
    "war_during_construction": col("war_during_construction"),
    "fire_during_construction": col("fire_during_construction"),
    "earthquake_during_construction": col("earthquake_during_construction"),
    "disaster_years": col("disaster_years"),

    "osmwikipedia": col("osmwikipedia"),
    "osmlink": col("osmlink"),
    "saints": master["saints_compiled"],
    "destroyed": master["damage_compiled"],
})

OUT_DIR.mkdir(parents=True, exist_ok=True)
out.to_csv(OUT_PATH, index=False)
print(f"✅ Wrote: {OUT_PATH}")
print("Rows (cathedrals):", len(out), "Columns:", out.shape[1])

# ============================================================
# WIKIDATA / WIKIPEDIA ENRICHMENT
# ============================================================

# ===============================
#  FULL CATHEDRAL STYLE PIPELINE
#  - Robust Wikipedia URL handling
#  - Wikipedia -> Wikidata QID (pageprops.wikibase_item)
#  - Wikidata -> architectural styles (P149) via WDQS SPARQL
#  - Preserve multiple styles (comma-separated)
#  - Add period ranges next to each style
#  - Save merged output as NEW CSV
# ===============================

import re
import time
from urllib.parse import urlparse, unquote, parse_qs
from bs4 import BeautifulSoup

import pandas as pd
import requests
from tqdm import tqdm

# -------------------------------
# Config
# -------------------------------
# These are created above by the base-data step.
INPUT_CSV = BASE_OUTPUT_CSV
OUTPUT_CSV = FINAL_OUTPUT_CSV

# -------------------------------
# 1) Load Data
# -------------------------------
df = pd.read_csv(INPUT_CSV)

candidate_cols = ["osmwikipedia_url", "osmwikipedia", "wikipedia", "wiki", "wikipedia_url"]
WIKI_COL = next((c for c in candidate_cols if c in df.columns), None)
if WIKI_COL is None:
    raise ValueError(f"No Wikipedia URL column found. Columns: {list(df.columns)}")

print("Using Wikipedia column:", WIKI_COL)

# -------------------------------
# 2) Robust Wikipedia URL helpers
# -------------------------------
def normalize_wikipedia_url(url: str) -> str | None:
    """
    Normalize Wikipedia URL:
    - add scheme if missing
    - convert mobile host (*.m.wikipedia.org) -> desktop
    - KEEP query string (critical for ?curid= and /w/index.php?title=)
    - drop fragments
    """
    if not isinstance(url, str) or not url.strip():
        return None
    url = url.strip()

    if url.startswith("//"):
        url = "https:" + url
    if not url.startswith("http"):
        url = "https://" + url.lstrip("/")

    try:
        u = urlparse(url)
    except Exception:
        return None

    host = (u.netloc or "").replace(".m.wikipedia.org", ".wikipedia.org")

    norm = f"{u.scheme}://{host}{u.path}"
    if u.query:
        norm += "?" + u.query
    return norm

def wikipedia_api_endpoint(wiki_url: str) -> str:
    u = urlparse(wiki_url)
    return f"{u.scheme}://{u.netloc}/w/api.php"

def extract_title_or_curid(wiki_url: str) -> dict:
    """
    Supports:
    - /wiki/Title
    - /w/index.php?title=Title
    - ?curid=12345
    """
    u = urlparse(wiki_url)
    qs = parse_qs(u.query or "")

    # ?curid=...
    if "curid" in qs and qs["curid"]:
        return {"pageids": qs["curid"][0]}

    # /w/index.php?title=...
    if "title" in qs and qs["title"]:
        title = unquote(qs["title"][0]).replace("_", " ")
        return {"titles": title}

    # /wiki/Title
    m = re.match(r"^/wiki/(.+)$", u.path or "")
    if m:
        title = unquote(m.group(1)).replace("_", " ")
        return {"titles": title}

    return {}

# -------------------------------
# 3) Wikipedia -> Wikidata QID
# -------------------------------
session = requests.Session()
session.headers.update({"User-Agent": "cathedral-style-research/1.1 (academic use)"})

def fetch_qid(wiki_url: str, session: requests.Session, sleep_s: float = 0.12, retries: int = 6) -> str | None:
    """
    Resolve a Wikipedia article URL -> Wikidata QID using:
      action=query&prop=pageprops&ppprop=wikibase_item
    Includes retries + backoff for 429/rate limits.
    """
    wiki_url = normalize_wikipedia_url(wiki_url)
    if not wiki_url:
        return None

    key = extract_title_or_curid(wiki_url)
    if not key:
        return None

    api = wikipedia_api_endpoint(wiki_url)
    params = {
        "action": "query",
        "format": "json",
        "prop": "pageprops",
        "ppprop": "wikibase_item",
        "redirects": 1,
        **key
    }

    for attempt in range(retries):
        try:
            r = session.get(api, params=params, timeout=30)

            # backoff if rate limited
            if r.status_code == 429:
                time.sleep(1.5 * (attempt + 1))
                continue

            r.raise_for_status()
            data = r.json()
            pages = data.get("query", {}).get("pages", {}) or {}
            for _, page in pages.items():
                qid = (page.get("pageprops") or {}).get("wikibase_item")
                if isinstance(qid, str) and qid.startswith("Q"):
                    return qid
            return None

        except Exception:
            time.sleep(0.6 * (attempt + 1))
        finally:
            time.sleep(sleep_s)

    return None

print("Resolving Wikipedia → Wikidata QIDs...")
df["osmwikipedia_url_norm"] = df[WIKI_COL].apply(normalize_wikipedia_url)

qid_cache: dict[str, str | None] = {}
qids = []
for url in tqdm(df["osmwikipedia_url_norm"].tolist(), desc="QID lookups"):
    if not url:
        qids.append(None)
        continue
    if url in qid_cache:
        qids.append(qid_cache[url])
        continue
    qid = fetch_qid(url, session=session)
    qid_cache[url] = qid
    qids.append(qid)

df["wikidata_qid"] = qids
print("Resolved QIDs:", df["wikidata_qid"].notna().sum(), "/", len(df))

# -------------------------------
# 4) Wikidata styles (P149) via WDQS
# -------------------------------
WDQS_URL = "https://query.wikidata.org/sparql"

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

def extract_qid_from_entity_uri(uri: str) -> str | None:
    m = re.search(r"/entity/(Q\d+)$", str(uri))
    return m.group(1) if m else None

def wdqs_get_styles(qid_list, session: requests.Session, sleep_s: float = 1.0) -> pd.DataFrame:
    """
    Batch query WDQS for architectural style (P149).
    Returns rows with columns: item_uri, styleLabel
    """
    values = " ".join(f"wd:{q}" for q in qid_list if isinstance(q, str) and q.startswith("Q"))
    if not values.strip():
        return pd.DataFrame(columns=["item_uri", "styleLabel"])

    sparql = f"""
    SELECT ?item ?styleLabel WHERE {{
      VALUES ?item {{ {values} }}
      OPTIONAL {{ ?item wdt:P149 ?style. }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """

    try:
        r = session.get(
            WDQS_URL,
            params={"query": sparql},
            headers={"Accept": "application/sparql-results+json"},
            timeout=60
        )
        if r.status_code == 429:
            # WDQS throttling
            time.sleep(4.0)
            return pd.DataFrame(columns=["item_uri", "styleLabel"])
        r.raise_for_status()
        data = r.json()

        rows = []
        for b in data.get("results", {}).get("bindings", []):
            rows.append({
                "item_uri": b["item"]["value"],
                "styleLabel": b.get("styleLabel", {}).get("value")
            })
        return pd.DataFrame(rows)

    except Exception:
        return pd.DataFrame(columns=["item_uri", "styleLabel"])
    finally:
        time.sleep(sleep_s)

print("Querying WDQS for P149 architectural styles...")
all_qids = sorted(set(q for q in df["wikidata_qid"].dropna().tolist() if isinstance(q, str)))
print("Unique QIDs:", len(all_qids))

BATCH_SIZE = 40  # smaller = more reliable
style_frames = []

for batch in tqdm(list(chunks(all_qids, BATCH_SIZE)), desc="WDQS batches"):
    style_frames.append(wdqs_get_styles(batch, session=session, sleep_s=1.0))

styles_long = pd.concat(style_frames, ignore_index=True) if style_frames else pd.DataFrame()
styles_long["wikidata_qid"] = styles_long["item_uri"].apply(extract_qid_from_entity_uri)

# Aggregate styles per QID (comma-separated, de-duplicated)
style_agg = (styles_long
             .dropna(subset=["styleLabel"])
             .groupby("wikidata_qid", as_index=False)
             .agg(style_wikidata=("styleLabel",
                                 lambda x: ", ".join(sorted(set(str(v).strip() for v in x if str(v).strip())))))
            )

df = df.merge(style_agg, on="wikidata_qid", how="left")

# -------------------------------
# 4B) FALLBACK: Wikipedia infobox "Architectural style"
#      (only when Wikidata P149 is missing)
# -------------------------------

def wiki_parse_api_endpoint(wiki_url: str) -> str:
    """
    Use Wikipedia API of the same language edition as the URL.
    """
    u = urlparse(wiki_url)
    host = (u.netloc or "").replace(".m.wikipedia.org", ".wikipedia.org")
    return f"{u.scheme}://{host}/w/api.php"

def fetch_wikitext_infobox_style(wiki_url: str, session: requests.Session, sleep_s: float = 0.2, retries: int = 4) -> str | None:
    """
    Pull raw wikitext for the page and parse the 'architectural_style' / 'architecture' infobox field.
    Works best for pages that use an Infobox (common for cathedrals).
    """
    wiki_url = normalize_wikipedia_url(wiki_url)
    if not wiki_url:
        return None

    key = extract_title_or_curid(wiki_url)
    if not key:
        return None

    api = wiki_parse_api_endpoint(wiki_url)

    params = {
        "action": "query",
        "format": "json",
        "formatversion": 2,
        "prop": "revisions",
        "rvprop": "content",
        "rvslots": "main",
        "redirects": 1,
        **key
    }

    for attempt in range(retries):
        try:
            r = session.get(api, params=params, timeout=45)
            if r.status_code == 429:
                time.sleep(1.5 * (attempt + 1))
                continue
            r.raise_for_status()
            data = r.json()

            pages = data.get("query", {}).get("pages", [])
            if not pages:
                return None

            revs = pages[0].get("revisions", [])
            if not revs:
                return None

            content = revs[0].get("slots", {}).get("main", {}).get("content", "")
            if not isinstance(content, str) or not content.strip():
                return None

            # Common infobox param keys:
            # | architectural_style = Gothic architecture
            # | architecture       = Gothic
            # | style              = ...
            # We'll search a few in order.
            keys = [
                "architectural_style",
                "architectural style",
                "architecture",
                "style",
            ]

            # Helper: extract value after "| key = value"
            def extract_param(text: str, key: str) -> str | None:
                # match like: | key = value (until newline or next | )
                # make it robust to spaces and underscores
                key_pat = re.escape(key).replace("\\_", "_")
                m = re.search(rf"\|\s*{key_pat}\s*=\s*(.+)", text, flags=re.IGNORECASE)
                if not m:
                    return None
                val = m.group(1).strip()
                # stop at start of next param if it got captured
                val = re.split(r"\n\|\s*[A-Za-z_ -]+\s*=", val)[0].strip()
                return val if val else None

            raw_val = None
            for k in keys:
                raw_val = extract_param(content, k)
                if raw_val:
                    break

            if not raw_val:
                return None

            # Clean wikitext:
            # - remove refs
            # - replace links [[Gothic architecture|Gothic]] -> Gothic
            # - remove templates {{...}}
            raw_val = re.sub(r"<ref[^>]*>.*?</ref>", "", raw_val, flags=re.DOTALL)
            raw_val = re.sub(r"<ref[^/]*/\s*>", "", raw_val)
            raw_val = re.sub(r"\{\{.*?\}\}", "", raw_val)  # simple template strip
            raw_val = re.sub(r"\[\[([^|\]]+)\|([^\]]+)\]\]", r"\2", raw_val)  # piped links -> display
            raw_val = re.sub(r"\[\[([^\]]+)\]\]", r"\1", raw_val)              # normal links
            raw_val = raw_val.replace("&nbsp;", " ")
            raw_val = re.sub(r"\s+", " ", raw_val).strip()

            # Split common separators into comma list
            # Some pages use <br>, "and", "/", ";"
            raw_val = raw_val.replace("<br />", ",").replace("<br/>", ",").replace("<br>", ",")
            raw_val = raw_val.replace(" / ", ", ")
            raw_val = raw_val.replace(";", ",")
            raw_val = re.sub(r"\s+and\s+", ", ", raw_val, flags=re.IGNORECASE)

            # Normalize comma spacing and dedupe
            parts = [p.strip() for p in raw_val.split(",") if p.strip()]
            parts = sorted(set(parts))
            cleaned = ", ".join(parts)

            return cleaned if cleaned else None

        except Exception:
            time.sleep(0.6 * (attempt + 1))
        finally:
            time.sleep(sleep_s)

    return None

# Create source column
df["style_source"] = None
df.loc[df["style_wikidata"].notna() & (df["style_wikidata"].astype(str).str.strip() != ""), "style_source"] = "wikidata_p149"

# Identify rows where we have QID but missing style from Wikidata
needs_fallback = df["wikidata_qid"].notna() & (
    df["style_wikidata"].isna() | (df["style_wikidata"].astype(str).str.strip() == "")
)

print("Rows needing Wikipedia infobox fallback:", needs_fallback.sum())

# Run fallback only for those rows
fallback_styles = []
for idx, row in tqdm(df[needs_fallback].iterrows(), total=needs_fallback.sum(), desc="Infobox style fallback"):
    wiki_url = row.get("osmwikipedia_url_norm") or row.get(WIKI_COL)
    style = fetch_wikitext_infobox_style(wiki_url, session=session)
    fallback_styles.append((idx, style))

# Write fallback styles into style_wikidata where missing
filled = 0
for idx, style in fallback_styles:
    if style:
        df.at[idx, "style_wikidata"] = style
        df.at[idx, "style_source"] = "wikipedia_infobox"
        filled += 1

# Mark remaining missing
still_missing = df["style_wikidata"].isna() | (df["style_wikidata"].astype(str).str.strip() == "")
df.loc[still_missing & df["style_source"].isna(), "style_source"] = "missing"

print(f"Fallback filled styles for {filled} rows.")
print("Styles still missing after fallback:", still_missing.sum())

# -------------------------------
# 4C) Wikidata materials used (P186) via WDQS
# -------------------------------

def wdqs_get_materials(qid_list, session: requests.Session, sleep_s: float = 1.0) -> pd.DataFrame:
    """
    Batch query WDQS for material used (P186).
    Returns: wikidata_qid, materialLabel
    """
    values = " ".join(f"wd:{q}" for q in qid_list if isinstance(q, str) and q.startswith("Q"))
    if not values.strip():
        return pd.DataFrame(columns=["item_uri", "materialLabel"])

    sparql = f"""
    SELECT ?item ?materialLabel WHERE {{
      VALUES ?item {{ {values} }}
      OPTIONAL {{ ?item wdt:P186 ?material. }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """
    try:
        r = session.get(
            WDQS_URL,
            params={"query": sparql},
            headers={"Accept": "application/sparql-results+json"},
            timeout=60
        )
        if r.status_code == 429:
            time.sleep(4.0)
            return pd.DataFrame(columns=["item_uri", "materialLabel"])
        r.raise_for_status()
        data = r.json()

        rows = []
        for b in data.get("results", {}).get("bindings", []):
            rows.append({
                "item_uri": b["item"]["value"],
                "materialLabel": b.get("materialLabel", {}).get("value")
            })
        return pd.DataFrame(rows)
    except Exception:
        return pd.DataFrame(columns=["item_uri", "materialLabel"])
    finally:
        time.sleep(sleep_s)

print("Querying WDQS for P186 materials...")
all_qids = sorted(set(q for q in df["wikidata_qid"].dropna().tolist() if isinstance(q, str)))

material_frames = []
for batch in tqdm(list(chunks(all_qids, BATCH_SIZE)), desc="WDQS material batches"):
    material_frames.append(wdqs_get_materials(batch, session=session, sleep_s=1.0))

materials_long = pd.concat(material_frames, ignore_index=True) if material_frames else pd.DataFrame()
materials_long["wikidata_qid"] = materials_long["item_uri"].apply(extract_qid_from_entity_uri)

material_agg = (materials_long
                .dropna(subset=["materialLabel"])
                .groupby("wikidata_qid", as_index=False)
                .agg(material_wikidata=("materialLabel",
                                        lambda x: ", ".join(sorted(set(str(v).strip() for v in x if str(v).strip())))))
               )

df = df.merge(material_agg, on="wikidata_qid", how="left")

# -------------------------------
# 4D) Wikipedia infobox fallback for materials
# -------------------------------

def fetch_wikitext_infobox_materials(wiki_url: str, session: requests.Session, sleep_s: float = 0.2, retries: int = 4) -> str | None:
    """
    Pull wikitext and parse infobox 'materials' / 'material' / 'building_material' / similar.
    """
    wiki_url = normalize_wikipedia_url(wiki_url)
    if not wiki_url:
        return None

    key = extract_title_or_curid(wiki_url)
    if not key:
        return None

    api = wiki_parse_api_endpoint(wiki_url)
    params = {
        "action": "query",
        "format": "json",
        "formatversion": 2,
        "prop": "revisions",
        "rvprop": "content",
        "rvslots": "main",
        "redirects": 1,
        **key
    }

    for attempt in range(retries):
        try:
            r = session.get(api, params=params, timeout=45)
            if r.status_code == 429:
                time.sleep(1.5 * (attempt + 1))
                continue
            r.raise_for_status()
            data = r.json()

            pages = data.get("query", {}).get("pages", [])
            if not pages:
                return None
            revs = pages[0].get("revisions", [])
            if not revs:
                return None

            content = revs[0].get("slots", {}).get("main", {}).get("content", "")
            if not isinstance(content, str) or not content.strip():
                return None

            keys = [
                "materials",
                "material",
                "building_material",
                "building material",
                "construction_material",
                "construction material",
                "material_used",
                "material used",
                "stone_used",
                "stone used",
                "metal_used",
                "metal used",
            ]

            def extract_param(text: str, key: str) -> str | None:
                key_pat = re.escape(key).replace("\\_", "_")
                m = re.search(rf"\|\s*{key_pat}\s*=\s*(.+)", text, flags=re.IGNORECASE)
                if not m:
                    return None
                val = m.group(1).strip()
                val = re.split(r"\n\|\s*[A-Za-z_ -]+\s*=", val)[0].strip()
                return val if val else None

            raw_val = None
            for k in keys:
                raw_val = extract_param(content, k)
                if raw_val:
                    break
            if not raw_val:
                return None

            # Clean wikitext
            raw_val = re.sub(r"<ref[^>]*>.*?</ref>", "", raw_val, flags=re.DOTALL)
            raw_val = re.sub(r"<ref[^/]*/\s*>", "", raw_val)
            raw_val = re.sub(r"\{\{.*?\}\}", "", raw_val)
            raw_val = re.sub(r"\[\[([^|\]]+)\|([^\]]+)\]\]", r"\2", raw_val)
            raw_val = re.sub(r"\[\[([^\]]+)\]\]", r"\1", raw_val)
            raw_val = raw_val.replace("&nbsp;", " ")
            raw_val = re.sub(r"\s+", " ", raw_val).strip()

            # Normalize separators to commas
            raw_val = raw_val.replace("<br />", ",").replace("<br/>", ",").replace("<br>", ",")
            raw_val = raw_val.replace(" / ", ", ")
            raw_val = raw_val.replace(";", ",")
            raw_val = re.sub(r"\s+and\s+", ", ", raw_val, flags=re.IGNORECASE)

            parts = [p.strip() for p in raw_val.split(",") if p.strip()]
            # De-dupe while preserving order
            seen = set()
            cleaned_parts = []
            for p in parts:
                keyp = p.lower()
                if keyp not in seen:
                    seen.add(keyp)
                    cleaned_parts.append(p)

            cleaned = ", ".join(cleaned_parts)
            return cleaned if cleaned else None

        except Exception:
            time.sleep(0.6 * (attempt + 1))
        finally:
            time.sleep(sleep_s)

    return None

# material_source
df["material_source"] = None
df.loc[df["material_wikidata"].notna() & (df["material_wikidata"].astype(str).str.strip() != ""), "material_source"] = "wikidata_p186"

needs_material_fallback = df["wikidata_qid"].notna() & (
    df["material_wikidata"].isna() | (df["material_wikidata"].astype(str).str.strip() == "")
)

print("Rows needing Wikipedia infobox materials fallback:", needs_material_fallback.sum())

fallback_materials = []
for idx, row in tqdm(df[needs_material_fallback].iterrows(), total=needs_material_fallback.sum(), desc="Infobox materials fallback"):
    wiki_url = row.get("osmwikipedia_url_norm") or row.get(WIKI_COL)
    mats = fetch_wikitext_infobox_materials(wiki_url, session=session)
    fallback_materials.append((idx, mats))

df["material_wikipedia_infobox"] = ""

filled_mat = 0
for idx, mats in fallback_materials:
    if mats:
        df.at[idx, "material_wikipedia_infobox"] = mats
        df.at[idx, "material_source"] = "wikipedia_infobox"
        filled_mat += 1

# final material (prefer wikidata; else wikipedia)
df["material_final"] = df["material_wikidata"]
mask_use_wiki = (df["material_final"].isna()) | (df["material_final"].astype(str).str.strip() == "")
df.loc[mask_use_wiki, "material_final"] = df.loc[mask_use_wiki, "material_wikipedia_infobox"]

still_missing_mat = (df["material_final"].isna()) | (df["material_final"].astype(str).str.strip() == "")
df.loc[still_missing_mat & df["material_source"].isna(), "material_source"] = "missing"

print(f"Material fallback filled {filled_mat} rows.")
print("Materials still missing after fallback:", still_missing_mat.sum())

# -------------------------------
# 4D2) Wikipedia date extraction + reconciliation (UPGRADED)
#      - Extract dates from infobox HTML
#      - Extract earliest-origin / current-structure clues from page text
#      - Handle English + Italian labels
#      - Handle exact years, numeric centuries, Roman numeral centuries,
#        and spelled-out English centuries
#      - Canonical Wikipedia start = earliest available start candidate
#      - Canonical Wikipedia end = infobox end if present, else text current end
#      - Replace ONLY when both original and Wikipedia values exist and differ
#      - Preserve originals + audit columns
# -------------------------------

DATE_CACHE = {}

START_LABEL_PRIORITY = [
    # English
    "groundbreaking",
    "broke ground",
    "construction began",
    "construction begins",
    "began",
    "founded",
    "foundation stone laid",
    "built",
    "building began",

    # Italian
    "inizio costruzione",
    "costruzione iniziata",
    "inizio dei lavori",
    "inizio lavori",
    "fondata",
    "fondazione",
    "costruita",
    "edificata",
]

END_LABEL_PRIORITY = [
    # English
    "completed",
    "completion date",
    "completion",
    "finished",
    "consecrated",
    "dedicated",
    "opened",

    # Italian
    "completamento",
    "completata",
    "fine costruzione",
    "fine dei lavori",
    "ultimazione",
    "consacrata",
    "dedicata",
    "inaugurata",
]

ENGLISH_ORDINAL_CENTURIES = {
    "first": 1,
    "second": 2,
    "third": 3,
    "fourth": 4,
    "fifth": 5,
    "sixth": 6,
    "seventh": 7,
    "eighth": 8,
    "ninth": 9,
    "tenth": 10,
    "eleventh": 11,
    "twelfth": 12,
    "thirteenth": 13,
    "fourteenth": 14,
    "fifteenth": 15,
    "sixteenth": 16,
    "seventeenth": 17,
    "eighteenth": 18,
    "nineteenth": 19,
    "twentieth": 20,
    "twenty-first": 21,
    "twenty first": 21,
}

ITALIAN_ORDINAL_CENTURIES = {
    "primo": 1,
    "secondo": 2,
    "terzo": 3,
    "quarto": 4,
    "quinto": 5,
    "sesto": 6,
    "settimo": 7,
    "ottavo": 8,
    "nono": 9,
    "decimo": 10,
    "undicesimo": 11,
    "dodicesimo": 12,
    "tredicesimo": 13,
    "quattordicesimo": 14,
    "quindicesimo": 15,
    "sedicesimo": 16,
    "diciassettesimo": 17,
    "diciottesimo": 18,
    "diciannovesimo": 19,
    "ventesimo": 20,
    "ventunesimo": 21,
}

ROMAN_MAP = {
    "I": 1, "V": 5, "X": 10, "L": 50,
    "C": 100, "D": 500, "M": 1000
}

def roman_to_int(r):
    total = 0
    prev = 0
    for ch in reversed(r.upper()):
        val = ROMAN_MAP.get(ch, 0)
        if val < prev:
            total -= val
        else:
            total += val
            prev = val
    return total if total > 0 else None

def century_to_year(c, mode="start"):
    """
    Convert a century number to a year boundary.
    start mode: 4th century -> 300
    end mode:   4th century -> 400
    """
    if c is None or c <= 0:
        return None
    return (c - 1) * 100 if mode == "start" else c * 100

def clean_year_value(text, mode="start") -> int | None:
    """
    Extract a usable year from messy text.

    Handles:
      - exact dates/years: 'December 27, 1892' -> 1892
      - numeric centuries: '6th century' -> 500 / 600
      - Italian numeric centuries: '6 secolo' -> 500 / 600
      - Roman numeral centuries: 'VI secolo', 'XVII secolo'
      - English ordinal centuries: 'fourth century', 'eleventh century'
      - Italian ordinal centuries: 'quarto secolo', 'undicesimo secolo'
    """
    if text is None:
        return None

    s = str(text).strip().lower()
    if not s:
        return None

    s = re.sub(r"\[[^\]]*\]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    # 1) Exact 3- or 4-digit year
    year_match = re.search(r"\b(\d{3,4})\b", s)
    if year_match:
        return int(year_match.group(1))

    # 2) Numeric English centuries: "6th century", "17th century"
    century_match = re.search(r"\b(\d{1,2})(st|nd|rd|th)\s+century\b", s)
    if century_match:
        c = int(century_match.group(1))
        return century_to_year(c, mode=mode)

    # 3) Numeric Italian centuries: "6 secolo", "6° secolo"
    century_it_numeric = re.search(r"\b(\d{1,2})\s*°?\s*secolo\b", s)
    if century_it_numeric:
        c = int(century_it_numeric.group(1))
        return century_to_year(c, mode=mode)

    # 4) Roman numeral Italian centuries: "vi secolo", "xvii secolo"
    roman_match = re.search(r"\b([ivxlcdm]+)\s+secolo\b", s)
    if roman_match:
        c = roman_to_int(roman_match.group(1))
        return century_to_year(c, mode=mode)

    # 5) English spelled-out centuries: "fourth century", "eleventh century"
    for word, c in ENGLISH_ORDINAL_CENTURIES.items():
        if re.search(rf"\b{re.escape(word)}\s+century\b", s):
            return century_to_year(c, mode=mode)

    # 6) Italian spelled-out centuries: "quarto secolo", "undicesimo secolo"
    for word, c in ITALIAN_ORDINAL_CENTURIES.items():
        if re.search(rf"\b{re.escape(word)}\s+secolo\b", s):
            return century_to_year(c, mode=mode)

    return None

def extract_infobox_dates_from_html(html: str) -> dict:
    """
    Parse the Wikipedia infobox from HTML and pull candidate date fields.
    """
    result = {
        "wiki_date_fields": {},
        "start_year_wikipedia_infobox": None,
        "end_year_wikipedia_infobox": None,
        "start_year_wikipedia_infobox_label": None,
        "end_year_wikipedia_infobox_label": None,
    }

    if not isinstance(html, str) or not html.strip():
        return result

    soup = BeautifulSoup(html, "html.parser")
    infobox = soup.find("table", class_=lambda c: c and "infobox" in c)

    if infobox is None:
        return result

    fields = {}

    for tr in infobox.find_all("tr"):
        th = tr.find("th")
        td = tr.find("td")
        if th is None or td is None:
            continue

        label = th.get_text(" ", strip=True).lower()
        value = td.get_text(" ", strip=True)

        if label and value:
            fields[label] = value

    result["wiki_date_fields"] = fields

    # Start year from infobox
    for wanted in START_LABEL_PRIORITY:
        for label, value in fields.items():
            if wanted in label:
                y = clean_year_value(value, mode="start")
                if y is not None:
                    result["start_year_wikipedia_infobox"] = y
                    result["start_year_wikipedia_infobox_label"] = label
                    break
        if result["start_year_wikipedia_infobox"] is not None:
            break

    # End year from infobox
    for wanted in END_LABEL_PRIORITY:
        for label, value in fields.items():
            if wanted in label:
                y = clean_year_value(value, mode="end")
                if y is not None:
                    result["end_year_wikipedia_infobox"] = y
                    result["end_year_wikipedia_infobox_label"] = label
                    break
        if result["end_year_wikipedia_infobox"] is not None:
            break

    return result

def extract_page_text_dates_from_html(html: str) -> dict:
    """
    Pull earliest-origin and current-structure clues from article text.

    Goal:
      - earliest historical origin (e.g. "dates back to the fourth century")
      - current/main structure start (e.g. "current basilica was built in the eleventh century")
      - current/main structure end or later reconstruction date
    """
    result = {
        "start_year_wikipedia_earliest": None,
        "start_year_wikipedia_current": None,
        "end_year_wikipedia_current": None,
        "has_major_reconstruction_wikipedia": False,
        "wikipedia_date_notes": [],
    }

    if not isinstance(html, str) or not html.strip():
        return result

    soup = BeautifulSoup(html, "html.parser")
    paragraphs = soup.select("p")
    text = " ".join(p.get_text(" ", strip=True) for p in paragraphs)
    text_l = re.sub(r"\s+", " ", text).strip().lower()

    # Earliest-origin patterns
    earliest_patterns = [
        r"dates back to the ([^.,;:]+)",
        r"dates to the ([^.,;:]+)",
        r"the original church dates back to the ([^.,;:]+)",
        r"the original church dates to the ([^.,;:]+)",
        r"the first church dates to the ([^.,;:]+)",
        r"the first cathedral dates to the ([^.,;:]+)",
        r"the original basilica dates to the ([^.,;:]+)",
        r"was founded in ([^.,;:]+)",
        r"founded in the ([^.,;:]+)",
        r"originates in the ([^.,;:]+)",
        r"built in the ([^.,;:]+)",
        r"built on the ruins of .*? in the ([^.,;:]+)",
        r"risale al ([^.,;:]+)",          # Italian: dates back to
        r"risale al(?:la)? ([^.,;:]+)",
        r"la chiesa originaria .*? ([^.,;:]+)",
        r"la prima chiesa .*? ([^.,;:]+)",
        r"la prima cattedrale .*? ([^.,;:]+)",
        r"fondata nel ([^.,;:]+)",
        r"fondata in ([^.,;:]+)",
    ]

    # Current structure / reconstruction patterns
    current_start_patterns = [
        r"current (?:church|cathedral|basilica) was built in the ([^.,;:]+)",
        r"present (?:church|cathedral|basilica) was built in the ([^.,;:]+)",
        r"current structure was built in the ([^.,;:]+)",
        r"rebuilt in the ([^.,;:]+)",
        r"reconstructed in the ([^.,;:]+)",
        r"partially rebuilt in the ([^.,;:]+)",
        r"the current cathedral is .*? ([^.,;:]+)",
        r"the current basilica was built in the ([^.,;:]+)",
        r"l'attuale (?:chiesa|cattedrale|basilica) .*? ([^.,;:]+)",
        r"attuale (?:chiesa|cattedrale|basilica) .*? ([^.,;:]+)",
        r"ricostruita nel ([^.,;:]+)",
        r"ricostruita in ([^.,;:]+)",
        r"riedificata nel ([^.,;:]+)",
        r"riedificata in ([^.,;:]+)",
    ]

    current_end_patterns = [
        r"completed in ([^.,;:]+)",
        r"finished in ([^.,;:]+)",
        r"completed by ([^.,;:]+)",
        r"consecrated in ([^.,;:]+)",
        r"dedicated in ([^.,;:]+)",
        r"partially rebuilt again in the ([^.,;:]+)",
        r"rebuilt again in the ([^.,;:]+)",
        r"completed again in the ([^.,;:]+)",
        r"ultimata nel ([^.,;:]+)",
        r"completata nel ([^.,;:]+)",
        r"consacrata nel ([^.,;:]+)",
        r"dedicata nel ([^.,;:]+)",
        r"ricostruita nuovamente nel ([^.,;:]+)",
    ]

    # Earliest origin: collect all candidates, take minimum
    earliest_candidates = []
    for pat in earliest_patterns:
        for m in re.finditer(pat, text_l):
            y = clean_year_value(m.group(1), mode="start")
            if y is not None:
                earliest_candidates.append(y)
                result["wikipedia_date_notes"].append(f"earliest:{m.group(0)}")

    if earliest_candidates:
        result["start_year_wikipedia_earliest"] = min(earliest_candidates)

    # Current/main structure start: collect candidates, take minimum
    current_start_candidates = []
    for pat in current_start_patterns:
        for m in re.finditer(pat, text_l):
            y = clean_year_value(m.group(1), mode="start")
            if y is not None:
                current_start_candidates.append(y)
                result["wikipedia_date_notes"].append(f"current_start:{m.group(0)}")

    if current_start_candidates:
        result["start_year_wikipedia_current"] = min(current_start_candidates)
        result["has_major_reconstruction_wikipedia"] = True

    # Current/main structure end: collect candidates, take maximum
    current_end_candidates = []
    for pat in current_end_patterns:
        for m in re.finditer(pat, text_l):
            y = clean_year_value(m.group(1), mode="end")
            if y is not None:
                current_end_candidates.append(y)
                result["wikipedia_date_notes"].append(f"current_end:{m.group(0)}")

    if current_end_candidates:
        result["end_year_wikipedia_current"] = max(current_end_candidates)

    return result

def fetch_wikipedia_date_data(
    wiki_url: str,
    session: requests.Session,
    sleep_s: float = 0.2,
    retries: int = 4
) -> dict:
    """
    Fetch the live Wikipedia page HTML and parse:
      - infobox dates
      - page-text origin/reconstruction dates
    Cached by normalized URL.
    """
    wiki_url = normalize_wikipedia_url(wiki_url)
    if not wiki_url:
        return {
            "wiki_date_fields": {},
            "start_year_wikipedia_infobox": None,
            "end_year_wikipedia_infobox": None,
            "start_year_wikipedia_infobox_label": None,
            "end_year_wikipedia_infobox_label": None,
            "start_year_wikipedia_earliest": None,
            "start_year_wikipedia_current": None,
            "end_year_wikipedia_current": None,
            "has_major_reconstruction_wikipedia": False,
            "wikipedia_date_notes": [],
        }

    if wiki_url in DATE_CACHE:
        return DATE_CACHE[wiki_url]

    headers = {"User-Agent": "cathedral-style-research/1.2 (academic use)"}

    for attempt in range(retries):
        try:
            r = session.get(wiki_url, headers=headers, timeout=30)
            if r.status_code == 429:
                time.sleep(1.5 * (attempt + 1))
                continue

            r.raise_for_status()

            infobox_data = extract_infobox_dates_from_html(r.text)
            text_data = extract_page_text_dates_from_html(r.text)

            parsed = {
                **infobox_data,
                **text_data,
            }

            DATE_CACHE[wiki_url] = parsed
            return parsed

        except Exception:
            time.sleep(0.6 * (attempt + 1))
        finally:
            time.sleep(sleep_s)

    fallback = {
        "wiki_date_fields": {},
        "start_year_wikipedia_infobox": None,
        "end_year_wikipedia_infobox": None,
        "start_year_wikipedia_infobox_label": None,
        "end_year_wikipedia_infobox_label": None,
        "start_year_wikipedia_earliest": None,
        "start_year_wikipedia_current": None,
        "end_year_wikipedia_current": None,
        "has_major_reconstruction_wikipedia": False,
        "wikipedia_date_notes": [],
    }
    DATE_CACHE[wiki_url] = fallback
    return fallback

print("Extracting Wikipedia dates...")

wiki_date_rows = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Wikipedia date extraction"):
    wiki_url = row.get("osmwikipedia_url_norm") or row.get(WIKI_COL)
    parsed = fetch_wikipedia_date_data(wiki_url, session=session)

    # Canonical Wikipedia start:
    # use earliest available candidate across text earliest, text current, infobox start
    start_candidates = [
        parsed.get("start_year_wikipedia_earliest"),
        parsed.get("start_year_wikipedia_current"),
        parsed.get("start_year_wikipedia_infobox"),
    ]
    start_candidates = [x for x in start_candidates if x is not None]
    start_year_wikipedia_final = min(start_candidates) if start_candidates else None

    # Canonical Wikipedia end:
    # prefer infobox end; otherwise use text current end
    end_year_wikipedia_final = (
        parsed.get("end_year_wikipedia_infobox")
        if parsed.get("end_year_wikipedia_infobox") is not None
        else parsed.get("end_year_wikipedia_current")
    )

    wiki_date_rows.append({
        "start_year_wikipedia_infobox": parsed.get("start_year_wikipedia_infobox"),
        "end_year_wikipedia_infobox": parsed.get("end_year_wikipedia_infobox"),
        "start_year_wikipedia_infobox_label": parsed.get("start_year_wikipedia_infobox_label"),
        "end_year_wikipedia_infobox_label": parsed.get("end_year_wikipedia_infobox_label"),

        "start_year_wikipedia_earliest": parsed.get("start_year_wikipedia_earliest"),
        "start_year_wikipedia_current": parsed.get("start_year_wikipedia_current"),
        "end_year_wikipedia_current": parsed.get("end_year_wikipedia_current"),
        "has_major_reconstruction_wikipedia": parsed.get("has_major_reconstruction_wikipedia"),

        "start_year_wikipedia_final": start_year_wikipedia_final,
        "end_year_wikipedia_final": end_year_wikipedia_final,

        "wiki_date_fields": str(parsed.get("wiki_date_fields", {})),
        "wikipedia_date_notes": " | ".join(parsed.get("wikipedia_date_notes", [])),
    })

wiki_dates_df = pd.DataFrame(wiki_date_rows, index=df.index)
df = pd.concat([df, wiki_dates_df], axis=1)

# Ensure base columns exist
if "start_year" not in df.columns:
    df["start_year"] = pd.NA
if "end_year" not in df.columns:
    df["end_year"] = pd.NA

# Normalize to nullable integer dtype to avoid future assignment warnings
year_cols = [
    "start_year",
    "end_year",
    "start_year_wikipedia_infobox",
    "end_year_wikipedia_infobox",
    "start_year_wikipedia_earliest",
    "start_year_wikipedia_current",
    "end_year_wikipedia_current",
    "start_year_wikipedia_final",
    "end_year_wikipedia_final",
]

for col in year_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

# Preserve originals for auditing
df["start_year_original"] = df["start_year"].copy()
df["end_year_original"] = df["end_year"].copy()

# Mismatch flags: only where BOTH original and final Wikipedia values exist and differ
start_valid = df["start_year"].notna() & df["start_year_wikipedia_final"].notna()
end_valid = df["end_year"].notna() & df["end_year_wikipedia_final"].notna()

df["start_year_mismatch_wikipedia"] = False
df.loc[start_valid, "start_year_mismatch_wikipedia"] = (
    df.loc[start_valid, "start_year"].to_numpy(dtype="int64") !=
    df.loc[start_valid, "start_year_wikipedia_final"].to_numpy(dtype="int64")
)

df["end_year_mismatch_wikipedia"] = False
df.loc[end_valid, "end_year_mismatch_wikipedia"] = (
    df.loc[end_valid, "end_year"].to_numpy(dtype="int64") !=
    df.loc[end_valid, "end_year_wikipedia_final"].to_numpy(dtype="int64")
)

# Replace ONLY when both exist and differ
start_replace_mask = df["start_year_mismatch_wikipedia"]
end_replace_mask = df["end_year_mismatch_wikipedia"]

df.loc[start_replace_mask, "start_year"] = df.loc[start_replace_mask, "start_year_wikipedia_final"].astype("Int64")
df.loc[end_replace_mask, "end_year"] = df.loc[end_replace_mask, "end_year_wikipedia_final"].astype("Int64")

# Source tracking
df["start_year_source"] = df.get("start_year_source", pd.NA)
df["end_year_source"] = df.get("end_year_source", pd.NA)

df.loc[start_replace_mask, "start_year_source"] = "wikipedia_final_replaced"
df.loc[end_replace_mask, "end_year_source"] = "wikipedia_final_replaced"

df.loc[
    df["start_year"].notna() & ~start_replace_mask & df["start_year_source"].isna(),
    "start_year_source"
] = "original_kept"

df.loc[
    df["end_year"].notna() & ~end_replace_mask & df["end_year_source"].isna(),
    "end_year_source"
] = "original_kept"

df.loc[df["start_year"].isna(), "start_year_source"] = "missing"
df.loc[df["end_year"].isna(), "end_year_source"] = "missing"

print("Wikipedia infobox start years found:", int(df["start_year_wikipedia_infobox"].notna().sum()))
print("Wikipedia earliest-origin start years found:", int(df["start_year_wikipedia_earliest"].notna().sum()))
print("Wikipedia current-structure start years found:", int(df["start_year_wikipedia_current"].notna().sum()))
print("Wikipedia final start years found:", int(df["start_year_wikipedia_final"].notna().sum()))
print("Wikipedia final end years found:", int(df["end_year_wikipedia_final"].notna().sum()))
print("start_year replaced from Wikipedia:", int(start_replace_mask.sum()))
print("end_year replaced from Wikipedia:", int(end_replace_mask.sum()))

# -------------------------------
# 4E) Precious materials detection (based on material_final)
# -------------------------------

PRECIOUS_PATTERNS = [
    (r"\bgold\b|\bgild(ed|ing)?\b", "gold"),
    (r"\bsilver\b", "silver"),
    (r"\bbronze\b", "bronze"),
    (r"\bcopper\b", "copper"),
    (r"\bivory\b", "ivory"),
    (r"\bjade\b", "jade"),
    (r"\blapis\b|\blapis lazuli\b", "lapis lazuli"),
    (r"\bonyx\b", "onyx"),
    (r"\bmalachite\b", "malachite"),
    (r"\bagate\b", "agate"),
    (r"\bamethyst\b", "amethyst"),
    (r"\bsapphire\b", "sapphire"),
    (r"\bruby\b", "ruby"),
    (r"\bemerald\b", "emerald"),
    (r"\bdiamond\b", "diamond"),
    (r"\bprecious\b|\bgem(stone)?s?\b", "gemstones (generic)"),
    # optional: treat high-status stone finishes as “luxury”
    (r"\bmarble\b", "marble"),
    (r"\bporphyry\b", "porphyry"),
    (r"\balabaster\b", "alabaster"),
]

def detect_precious(material_text: str) -> str:
    if not isinstance(material_text, str) or not material_text.strip():
        return ""
    s = material_text.lower()
    hits = []
    for pat, label in PRECIOUS_PATTERNS:
        if re.search(pat, s, flags=re.IGNORECASE):
            hits.append(label)
    # de-dupe preserve order
    seen = set()
    out = []
    for h in hits:
        if h not in seen:
            seen.add(h)
            out.append(h)
    return ", ".join(out)

df["precious_materials_found"] = df["material_final"].apply(detect_precious)
df["has_precious_materials"] = df["precious_materials_found"].apply(lambda x: bool(isinstance(x, str) and x.strip()))

# -------------------------------
# 4F) HEIGHT (Wikidata P2048) -> fill missing df["height_m"] with plausible Wikidata height (meters)
#     - Treat blanks/whitespace ("   "), empty strings, and non-numeric as missing
#     - Keep existing numeric height_m values as-is
#     - Only use Wikidata values within a plausible range to avoid outliers
# -------------------------------

def wdqs_get_heights(qid_list, session: requests.Session, sleep_s: float = 1.0) -> pd.DataFrame:
    """
    Batch query WDQS for height (P2048).

    Returns rows:
      - wikidata_qid
      - height_m_wikidata (converted to meters where unit is known)
    """
    values = " ".join(f"wd:{q}" for q in qid_list if isinstance(q, str) and q.startswith("Q"))
    if not values.strip():
        return pd.DataFrame(columns=["wikidata_qid", "height_m_wikidata"])

    sparql = f"""
    SELECT ?item ?heightAmount ?heightUnit WHERE {{
      VALUES ?item {{ {values} }}
      OPTIONAL {{
        ?item p:P2048/psv:P2048 ?heightNode .
        ?heightNode wikibase:quantityAmount ?heightAmount .
        ?heightNode wikibase:quantityUnit ?heightUnit .
      }}
    }}
    """

    try:
        r = session.get(
            WDQS_URL,
            params={"query": sparql},
            headers={"Accept": "application/sparql-results+json"},
            timeout=60
        )
        if r.status_code == 429:
            time.sleep(4.0)
            return pd.DataFrame(columns=["wikidata_qid", "height_m_wikidata"])

        r.raise_for_status()
        data = r.json()

        rows = []
        for b in data.get("results", {}).get("bindings", []):
            qid = extract_qid_from_entity_uri(b["item"]["value"])
            amt = b.get("heightAmount", {}).get("value")
            unit_uri = b.get("heightUnit", {}).get("value")

            if not qid or amt is None:
                continue

            try:
                amt = float(amt)
            except Exception:
                continue

            unit_qid = extract_qid_from_entity_uri(unit_uri) if unit_uri else None

            # Convert to meters (most are meters: Q11573)
            if unit_qid in (None, "Q11573"):          # meter or unspecified
                h_m = amt
            elif unit_qid == "Q3710":                 # foot
                h_m = amt * 0.3048
            elif unit_qid == "Q828224":               # kilometre
                h_m = amt * 1000.0
            elif unit_qid == "Q174728":               # centimetre
                h_m = amt / 100.0
            elif unit_qid == "Q174789":               # millimetre
                h_m = amt / 1000.0
            else:
                continue  # unknown unit -> skip

            rows.append({"wikidata_qid": qid, "height_m_wikidata": h_m})

        return pd.DataFrame(rows)

    except Exception:
        return pd.DataFrame(columns=["wikidata_qid", "height_m_wikidata"])
    finally:
        time.sleep(sleep_s)

print("Querying WDQS for height (P2048)...")
all_qids_for_height = sorted(set(
    q for q in df["wikidata_qid"].dropna().tolist()
    if isinstance(q, str) and q.startswith("Q")
))

height_frames = []
for batch in tqdm(list(chunks(all_qids_for_height, BATCH_SIZE)), desc="WDQS height batches"):
    height_frames.append(wdqs_get_heights(batch, session=session, sleep_s=1.0))

heights_long = pd.concat(height_frames, ignore_index=True) if height_frames else pd.DataFrame()

# --- Plausibility filter (meters) ---
# Keeps out extreme/incorrect values (e.g., km-scale heights).
PLAUS_MIN, PLAUS_MAX = 10, 250
heights_long["height_m_wikidata"] = pd.to_numeric(heights_long.get("height_m_wikidata"), errors="coerce")
heights_long = heights_long.dropna(subset=["height_m_wikidata"])
heights_long = heights_long[heights_long["height_m_wikidata"].between(PLAUS_MIN, PLAUS_MAX)]

# If multiple plausible heights exist for the same item, keep the MAX plausible value
height_agg = (
    heights_long
    .groupby("wikidata_qid", as_index=False)
    .agg(height_m_wikidata=("height_m_wikidata", "max"))
)

df = df.merge(height_agg, on="wikidata_qid", how="left")

# --- Normalize existing height_m and fill missing/blank with Wikidata ---
# Treat whitespace-only strings as missing, then coerce to numeric.
if "height_m" not in df.columns:
    df["height_m"] = pd.NA

height_m_raw = df["height_m"]

# If height_m is a string column (or mixed), strip whitespace and blank->NA
height_m_clean = height_m_raw.astype("string").str.strip()
height_m_clean = height_m_clean.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})

# Convert to numeric (non-numeric -> NaN)
df["height_m"] = pd.to_numeric(height_m_clean, errors="coerce")

# Fill only where height_m is missing AND we have a plausible Wikidata height
mask_fill_height = df["height_m"].isna() & df["height_m_wikidata"].notna()
df.loc[mask_fill_height, "height_m"] = df.loc[mask_fill_height, "height_m_wikidata"]

# Audit column: explain where the final height_m came from
# - wikidata_p2048_filled_missing: height_m was missing/blank/non-numeric and got filled
# - original_kept: height_m already had a numeric value (kept as-is)
# - missing: still missing after attempting Wikidata fill
df["height_source"] = df.get("height_source", pd.NA)
df.loc[mask_fill_height, "height_source"] = "wikidata_p2048_filled_missing"
df.loc[~df["height_m"].isna() & ~mask_fill_height, "height_source"] = "original_kept"
df.loc[df["height_m"].isna(), "height_source"] = "missing"

print("Heights filled from plausible Wikidata P2048:", int(mask_fill_height.sum()))
print("height_m now present:", int(df["height_m"].notna().sum()), "/", len(df))


# -------------------------------
# 4G) SURFACE AREA (Wikidata P2046) -> fill missing df["surface_m2"] with plausible Wikidata area (m²)
#     - Treat blanks/whitespace ("   "), empty strings, and non-numeric as missing
#     - Keep existing numeric surface_m2 values as-is
#     - Only use Wikidata values within a plausible range to avoid outliers
# -------------------------------

def wdqs_get_surface_areas(qid_list, session: requests.Session, sleep_s: float = 1.0) -> pd.DataFrame:
    """
    Batch query WDQS for area (P2046).

    Returns rows:
      - wikidata_qid
      - surface_m2_wikidata (converted to square meters where unit is known)
    """
    values = " ".join(f"wd:{q}" for q in qid_list if isinstance(q, str) and q.startswith("Q"))
    if not values.strip():
        return pd.DataFrame(columns=["wikidata_qid", "surface_m2_wikidata"])

    sparql = f"""
    SELECT ?item ?areaAmount ?areaUnit WHERE {{
      VALUES ?item {{ {values} }}
      OPTIONAL {{
        ?item p:P2046/psv:P2046 ?areaNode .
        ?areaNode wikibase:quantityAmount ?areaAmount .
        ?areaNode wikibase:quantityUnit ?areaUnit .
      }}
    }}
    """

    try:
        r = session.get(
            WDQS_URL,
            params={"query": sparql},
            headers={"Accept": "application/sparql-results+json"},
            timeout=60
        )
        if r.status_code == 429:
            time.sleep(4.0)
            return pd.DataFrame(columns=["wikidata_qid", "surface_m2_wikidata"])

        r.raise_for_status()
        data = r.json()

        rows = []
        for b in data.get("results", {}).get("bindings", []):
            qid = extract_qid_from_entity_uri(b["item"]["value"])
            amt = b.get("areaAmount", {}).get("value")
            unit_uri = b.get("areaUnit", {}).get("value")

            if not qid or amt is None:
                continue

            try:
                amt = float(amt)
            except Exception:
                continue

            unit_qid = extract_qid_from_entity_uri(unit_uri) if unit_uri else None

            # Convert to square meters
            # Common units in Wikidata quantities:
            #  - square metre: Q25343
            #  - square kilometre: Q712226
            #  - square centimetre: Q48440
            #  - hectare: Q35852
            #  - acre: Q35838
            # If unit is missing/unspecified, we assume m² (often true in practice, but not guaranteed).
            if unit_qid in (None, "Q25343"):           # square metre or unspecified
                a_m2 = amt
            elif unit_qid == "Q712226":                # square kilometre
                a_m2 = amt * 1_000_000.0
            elif unit_qid == "Q35852":                 # hectare
                a_m2 = amt * 10_000.0
            elif unit_qid == "Q35838":                 # acre
                a_m2 = amt * 4046.8564224
            elif unit_qid == "Q48440":                 # square centimetre
                a_m2 = amt / 10_000.0
            else:
                continue  # unknown unit -> skip

            rows.append({"wikidata_qid": qid, "surface_m2_wikidata": a_m2})

        return pd.DataFrame(rows)

    except Exception:
        return pd.DataFrame(columns=["wikidata_qid", "surface_m2_wikidata"])
    finally:
        time.sleep(sleep_s)

print("Querying WDQS for surface area (P2046)...")
all_qids_for_surface = sorted(set(
    q for q in df["wikidata_qid"].dropna().tolist()
    if isinstance(q, str) and q.startswith("Q")
))

surface_frames = []
for batch in tqdm(list(chunks(all_qids_for_surface, BATCH_SIZE)), desc="WDQS surface batches"):
    surface_frames.append(wdqs_get_surface_areas(batch, session=session, sleep_s=1.0))

surfaces_long = pd.concat(surface_frames, ignore_index=True) if surface_frames else pd.DataFrame()

# --- Plausibility filter (m²) ---
# Cathedrals: footprint/floor areas are usually in the hundreds to tens of thousands m².
# This range prevents obvious unit/scale mistakes (e.g., km² or tiny cm² values).
SURF_MIN, SURF_MAX = 50, 500_000
surfaces_long["surface_m2_wikidata"] = pd.to_numeric(surfaces_long.get("surface_m2_wikidata"), errors="coerce")
surfaces_long = surfaces_long.dropna(subset=["surface_m2_wikidata"])
surfaces_long = surfaces_long[surfaces_long["surface_m2_wikidata"].between(SURF_MIN, SURF_MAX)]

# If multiple plausible areas exist for the same item, keep the MAX plausible value
surface_agg = (
    surfaces_long
    .groupby("wikidata_qid", as_index=False)
    .agg(surface_m2_wikidata=("surface_m2_wikidata", "max"))
)

df = df.merge(surface_agg, on="wikidata_qid", how="left")

# --- Normalize existing surface_m2 and fill missing/blank with Wikidata ---
if "surface_m2" not in df.columns:
    df["surface_m2"] = pd.NA

surface_raw = df["surface_m2"]

surface_clean = surface_raw.astype("string").str.strip()
surface_clean = surface_clean.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})

df["surface_m2"] = pd.to_numeric(surface_clean, errors="coerce")

mask_fill_surface = df["surface_m2"].isna() & df["surface_m2_wikidata"].notna()
df.loc[mask_fill_surface, "surface_m2"] = df.loc[mask_fill_surface, "surface_m2_wikidata"]

# Audit column: explain where surface_m2 came from
df["surface_source"] = df.get("surface_source", pd.NA)
df.loc[mask_fill_surface, "surface_source"] = "wikidata_p2046_filled_missing"
df.loc[~df["surface_m2"].isna() & ~mask_fill_surface, "surface_source"] = "original_kept"
df.loc[df["surface_m2"].isna(), "surface_source"] = "missing"

print("Surface areas filled from plausible Wikidata P2046:", int(mask_fill_surface.sum()))
print("surface_m2 now present:", int(df["surface_m2"].notna().sum()), "/", len(df))

# OPTIONAL: recompute volume_proxy if both exist
if "height_m" in df.columns:
    df["volume_proxy"] = df["surface_m2"] * pd.to_numeric(df["height_m"], errors="coerce")



# -------------------------------
# 5) Add period ranges in brackets (comma-separated)
# -------------------------------
STYLE_PERIOD_RULES = [
    (r"\bpre-romanesque\b", "c. 500–1000"),
    (r"\bcarolingian\b", "c. 750–900"),
    (r"\bottonian\b", "c. 950–1050"),
    (r"\bromanesque\b", "c. 1000–1200"),
    (r"\bearly gothic\b", "c. 1140–1200"),
    (r"\bhigh gothic\b", "c. 1200–1300"),
    (r"\brayonnant\b", "c. 1230–1350"),
    (r"\bflamboyant\b", "c. 1350–1550"),
    (r"\bgothic\b", "c. 1140–1600"),
    (r"\brenaissance\b", "c. 1400–1600"),
    (r"\bmannerism\b", "c. 1520–1600"),
    (r"\bbaroque\b", "c. 1600–1750"),
    (r"\brococo\b", "c. 1730–1780"),
    (r"\bneoclassical\b", "c. 1750–1850"),
    (r"\bclassicism\b", "c. 1750–1850"),
    (r"\bneo-?gothic\b", "c. 1740–1900"),
    (r"\bgothic revival\b", "c. 1740–1900"),
    (r"\bhistoricism\b", "c. 1800–1910"),
    (r"\bneo-?byzantine\b", "c. 1850–1930"),
    (r"\bbyzantine\b", "c. 330–1453"),
    (r"\bmodern\b", "c. 1900–present"),
    (r"\bmodernist\b", "c. 1900–present"),
    (r"\binternational style\b", "c. 1920–1970"),
    (r"\bbrutalis(t|m)\b", "c. 1950–1980"),
]

def get_period(label: str) -> str | None:
    s = label.strip().lower()
    for pat, period in STYLE_PERIOD_RULES:
        if re.search(pat, s, flags=re.IGNORECASE):
            return period
    return None

def add_periods(style_cell) -> str:
    """
    Input:  "Gothic architecture, Romanesque architecture"
    Output: "Gothic architecture (c. 1140–1600), Romanesque architecture (c. 1000–1200)"
    """ 
    if not isinstance(style_cell, str) or not style_cell.strip():
        return ""
    parts = [p.strip() for p in style_cell.split(",") if p.strip()]
    out = []
    for p in parts:
        per = get_period(p)
        out.append(f"{p} ({per})" if per else p)
    return ", ".join(out)

df["style_wikidata_with_periods"] = df["style_wikidata"].apply(add_periods)


# -------------------------------
# 6) Optional compatibility-safe final recomputation
# -------------------------------
def final_period_bucket(y):
    if pd.isna(y):
        return np.nan
    y = float(y)
    if y < 900:
        return "700–899"
    if y < 1100:
        return "900–1099"
    if y < 1300:
        return "1100–1299"
    if y <= 1500:
        return "1300–1500"
    return "post-1500"

if RECOMPUTE_DERIVED_AFTER_WIKI_DATE_UPDATES:
    # This is intentionally optional because your original two-notebook pipeline did not
    # recompute these fields after Wikipedia date replacements. Turning it on makes the
    # final dependent columns internally consistent with the final start/end years.
    df["start_year"] = pd.to_numeric(df.get("start_year"), errors="coerce")
    df["end_year"] = pd.to_numeric(df.get("end_year"), errors="coerce")
    df["duration_years"] = df["end_year"] - df["start_year"]
    df["period_bucket"] = df["start_year"].map(final_period_bucket)

    if "surface_m2" in df.columns and "height_m" in df.columns:
        df["surface_m2"] = pd.to_numeric(df["surface_m2"], errors="coerce")
        df["height_m"] = pd.to_numeric(df["height_m"], errors="coerce")
        df["volume_proxy"] = df["surface_m2"] * df["height_m"]
        if "m3" in df.columns:
            df["m3"] = df["volume_proxy"]

    # Recompute disaster-window counts from the original disaster table, if present.
    if 'dis_path' in globals() and dis_path.exists() and "osmid" in df.columns:
        dis_final = read_csv_safely(dis_path, low_memory=False)
        dis_final.columns = [c.strip().lower() for c in dis_final.columns]
        if "osmid" in dis_final.columns:
            dis_final["osmid"] = normalize_osmid(dis_final["osmid"])
            dis_final["year"] = pd.to_numeric(dis_final.get("year"), errors="coerce")
            df["osmid"] = normalize_osmid(df["osmid"])

            tmp_final = df[["osmid", "start_year", "end_year"]].merge(dis_final, on="osmid", how="left")
            tmp_final["in_window"] = (
                tmp_final["year"].notna() & tmp_final["start_year"].notna() & tmp_final["end_year"].notna() &
                (tmp_final["year"] >= tmp_final["start_year"]) &
                (tmp_final["year"] <= tmp_final["end_year"])
            ).astype(int)

            def boolcol_final(frame, name):
                if name not in frame.columns:
                    return pd.Series(False, index=frame.index)
                return truthy_series(frame[name])

            tmp_final["war_in_window"] = ((tmp_final["in_window"] == 1) & boolcol_final(tmp_final, "war")).astype(int)
            tmp_final["fire_in_window"] = ((tmp_final["in_window"] == 1) & boolcol_final(tmp_final, "fire")).astype(int)
            tmp_final["eq_in_window"] = ((tmp_final["in_window"] == 1) & boolcol_final(tmp_final, "earthquake")).astype(int)

            window_final = (
                tmp_final.groupby("osmid", as_index=False)
                .agg(
                    disasters_during_construction=("in_window", "sum"),
                    war_during_construction=("war_in_window", "sum"),
                    fire_during_construction=("fire_in_window", "sum"),
                    earthquake_during_construction=("eq_in_window", "sum"),
                )
            )
            # Replace old window columns with recomputed values.
            df = df.drop(
                columns=[
                    "disasters_during_construction",
                    "war_during_construction",
                    "fire_during_construction",
                    "earthquake_during_construction",
                ],
                errors="ignore",
            ).merge(window_final, on="osmid", how="left")

# -------------------------------
# 7) Save new merged file
# -------------------------------
df.to_csv(OUTPUT_CSV, index=False)

print("\n✅ Done.")
print("Saved:", OUTPUT_CSV)
print("Rows:", len(df))
print("QIDs resolved:", df["wikidata_qid"].notna().sum())
print("Styles found (P149 present):", df["style_wikidata"].notna().sum())

# Optional: show a few missing-QID examples
print("\nExamples missing QID:")
print(df[df["wikidata_qid"].isna()][[WIKI_COL]].head(10).to_string(index=False))

# Optional preview:
df[[WIKI_COL, "wikidata_qid", "style_wikidata", "style_wikidata_with_periods"]].head(12)

# ===============================
# FINAL COLUMN INSPECTION
# Replaces Cathdral_Filter.ipynb
# ===============================
print("\nFinal columns:")
for col_name in df.columns:
    print(col_name)


/var/folders/8_/ww6tchkd18sdf8vp6yh6l11w0000gn/T/ipykernel_35612/1573861734.py:173: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({
/var/folders/8_/ww6tchkd18sdf8vp6yh6l11w0000gn/T/ipykernel_35612/1573861734.py:199: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({
/var/folders/8_/ww6tchkd18sdf8vp6yh6l11w0000gn/T/ipykernel_35612/1573861734.py:474: FutureWarning: Dow

✅ Wrote: /Users/jeeveshattri/Desktop/Monuments_Of_Power/Background_Data/cathedral_master.csv
Rows (cathedrals): 313 Columns: 33
Using Wikipedia column: osmwikipedia
Resolving Wikipedia → Wikidata QIDs...


QID lookups: 100%|██████████| 313/313 [26:41<00:00,  5.12s/it] 


Resolved QIDs: 274 / 313
Querying WDQS for P149 architectural styles...
Unique QIDs: 274


WDQS batches: 100%|██████████| 7/7 [00:08<00:00,  1.19s/it]


...

Wikipedia infobox start years found: 152
Wikipedia earliest-origin start years found: 76
Wikipedia current-structure start years found: 23
Wikipedia final start years found: 161
Wikipedia final end years found: 150
start_year replaced from Wikipedia: 127
end_year replaced from Wikipedia: 132
Querying WDQS for height (P2048)...


WDQS height batches: 100%|██████████| 7/7 [00:07<00:00,  1.09s/it]


Heights filled from plausible Wikidata P2048: 23
height_m now present: 78 / 313
Querying WDQS for surface area (P2046)...


WDQS surface batches: 100%|██████████| 7/7 [00:07<00:00,  1.08s/it]

Surface areas filled from plausible Wikidata P2046: 7
surface_m2 now present: 63 / 313

✅ Done.
Saved: /Users/jeeveshattri/Desktop/Monuments_Of_Power/Background_Data/cathedral_master_with_wikidata_styles_with_periods.csv
Rows: 313
QIDs resolved: 274
Styles found (P149 present): 271

Examples missing QID:
                                                   osmwikipedia
it.wikipedia.org/wiki/Cattedrale_di_Santa_Maria_Assunta_(Rieti)
        fr.wikipedia.org/wiki/Cathédrale_Saint-Maurice_d'Angers
                          it.wikipedia.org/wiki/Duomo_di_Urbino
               it.wikipedia.org/wiki/Duomo_di_Città_di_Castello
          fr.wikipedia.org/wiki/Cathédrale_Notre-Dame_de_Senlis
                              de.wikipedia.org/wiki/Mainzer_Dom
                           it.wikipedia.org/wiki/Duomo_di_Prato
           fr.wikipedia.org/wiki/Cathédrale_Notre-Dame_de_Paris
                               de.wikipedia.org/wiki/Bremer_Dom
                      en.wikipedia.org/wiki/Worcester_

In [3]:
# ============================================================
# CATHEDRAL TARGETED RESEARCH + FINAL EXPORT CLEANUP
# ============================================================
# Run this AFTER cathedral_full_pipeline.py.
#
# Purpose:
#   1) Do NOT compare against the old final CSV.
#   2) Re-research unstable/missing web-enrichment fields using Wikipedia/Wikidata.
#   3) Apply targeted source-checked corrections for the few important rows that drifted.
#   4) Add study_id and elevation_m.
#   5) Export ONLY the columns that existed in the original cathedral_final.csv.
#
# Output:
#   cathedral_final_researched_clean.csv
#   cathedral_targeted_research_audit.csv
#
# Dependencies:
#   pip install pandas numpy requests tqdm
# ============================================================

from __future__ import annotations

from pathlib import Path
from urllib.parse import urlparse, unquote, parse_qs
import json
import math
import re
import time
from typing import Any

import numpy as np
import pandas as pd
import requests
from tqdm import tqdm

# ------------------------------------------------------------
# 0) Paths
# ------------------------------------------------------------
PROJECT_DIR = Path("/Users/jeeveshattri/Desktop/Monuments_Of_Power/Background_Data")

# Prefer the normal output name from the full pipeline.
INPUT_CANDIDATES = [
    PROJECT_DIR / "cathedral_master_with_wikidata_styles_with_periods.csv",
    PROJECT_DIR / "cathedral_master_with_wikidata_styles_with_periods(2).csv",
]

INPUT_CSV = next((p for p in INPUT_CANDIDATES if p.exists()), INPUT_CANDIDATES[0])
OUTPUT_CSV = PROJECT_DIR / "cathedral_final_researched_clean.csv"
AUDIT_CSV = PROJECT_DIR / "cathedral_targeted_research_audit.csv"
ELEVATION_CACHE_CSV = PROJECT_DIR / "cathedral_elevation_cache.csv"

# If False, this preserves the original notebook behavior where duration/period_bucket
# were not always recomputed after Wikipedia date changes.
# If True, dates and durations become internally consistent after targeted corrections.
RECOMPUTE_DURATION_AFTER_TARGETED_DATE_FIXES = False

# ------------------------------------------------------------
# 1) Original final-column layout
# ------------------------------------------------------------
ORIGINAL_FINAL_COLUMNS = [
    "study_id",
    "cathedral_name",
    "city",
    "country",
    "lat",
    "lon",
    "start_year",
    "end_year",
    "duration_years",
    "period_bucket",
    "surface_m2",
    "height_m",
    "volume_proxy",
    "completed",
    "not_collapsed",
    "n_disasters",
    "any_war",
    "any_fire",
    "any_earthquake",
    "any_other_natural",
    "any_other",
    "first_disaster_year",
    "last_disaster_year",
    "disasters_during_construction",
    "war_during_construction",
    "fire_during_construction",
    "earthquake_during_construction",
    "disaster_years",
    "saints",
    "destroyed",
    "style_wikidata",
    "style_source",
    "material_final",
    "start_year_wikipedia_infobox",
    "end_year_wikipedia_infobox",
    "start_year_wikipedia_infobox_label",
    "end_year_wikipedia_infobox_label",
    "start_year_wikipedia_earliest",
    "start_year_wikipedia_current",
    "end_year_wikipedia_current",
    "has_major_reconstruction_wikipedia",
    "start_year_wikipedia_final",
    "end_year_wikipedia_final",
    "wikipedia_date_notes",
    "start_year_original",
    "end_year_original",
    "start_year_mismatch_wikipedia",
    "end_year_mismatch_wikipedia",
    "start_year_source",
    "end_year_source",
    "height_m_wikidata",
    "height_source",
    "surface_m2_wikidata",
    "surface_source",
    "style_wikidata_with_periods",
    "elevation_m",
]

# ------------------------------------------------------------
# 2) Utilities
# ------------------------------------------------------------
def first_existing_col(df: pd.DataFrame, candidates: list[str]) -> str | None:
    for c in candidates:
        if c in df.columns:
            return c
    return None


def to_num(x: Any) -> float:
    return pd.to_numeric(x, errors="coerce")


def normalize_wikipedia_url(url: Any) -> str | None:
    if pd.isna(url):
        return None
    url = str(url).strip()
    if not url or url.lower() in {"nan", "none"}:
        return None
    if url.startswith("//"):
        url = "https:" + url
    if not url.startswith("http"):
        url = "https://" + url.lstrip("/")
    try:
        u = urlparse(url)
    except Exception:
        return None
    host = (u.netloc or "").replace(".m.wikipedia.org", ".wikipedia.org")
    norm = f"{u.scheme}://{host}{u.path}"
    if u.query:
        norm += "?" + u.query
    return norm


def wikipedia_language(wiki_url: Any) -> str:
    url = normalize_wikipedia_url(wiki_url)
    if not url:
        return "en"
    host = urlparse(url).netloc
    first = host.split(".")[0]
    return first if first and first != "www" else "en"


def extract_wikipedia_title(wiki_url: Any) -> str | None:
    url = normalize_wikipedia_url(wiki_url)
    if not url:
        return None
    u = urlparse(url)
    qs = parse_qs(u.query or "")
    if "title" in qs and qs["title"]:
        return unquote(qs["title"][0]).replace("_", " ")
    m = re.match(r"^/wiki/(.+)$", u.path or "")
    if m:
        return unquote(m.group(1)).replace("_", " ")
    return None


def period_bucket(y: Any) -> str | float:
    y = pd.to_numeric(y, errors="coerce")
    if pd.isna(y):
        return np.nan
    y = int(y)
    if 700 <= y <= 899:
        return "700–899"
    if 900 <= y <= 1099:
        return "900–1099"
    if 1100 <= y <= 1299:
        return "1100–1299"
    if 1300 <= y <= 1500:
        return "1300–1500"
    if y > 1500:
        return "post-1500"
    return np.nan


def row_mask(df: pd.DataFrame, name: str, city: str | None = None, country: str | None = None) -> pd.Series:
    mask = df["cathedral_name"].astype(str).str.casefold().eq(name.casefold())
    if city is not None and "city" in df.columns:
        mask &= df["city"].astype(str).str.casefold().eq(city.casefold())
    if country is not None and "country" in df.columns:
        mask &= df["country"].astype(str).str.casefold().eq(country.casefold())
    return mask


session = requests.Session()
session.headers.update({"User-Agent": "cathedral-targeted-research-cleanup/1.0 academic-use"})

# ------------------------------------------------------------
# 3) Wikidata QID repair for rows where language Wikipedia resolution failed
# ------------------------------------------------------------
def fetch_qid_from_wikipedia_url(wiki_url: Any, retries: int = 4) -> str | None:
    wiki_url = normalize_wikipedia_url(wiki_url)
    if not wiki_url:
        return None

    u = urlparse(wiki_url)
    api = f"{u.scheme}://{u.netloc}/w/api.php"
    qs = parse_qs(u.query or "")
    params = {
        "action": "query",
        "format": "json",
        "prop": "pageprops",
        "ppprop": "wikibase_item",
        "redirects": 1,
    }

    if "curid" in qs and qs["curid"]:
        params["pageids"] = qs["curid"][0]
    else:
        title = extract_wikipedia_title(wiki_url)
        if not title:
            return None
        params["titles"] = title

    for attempt in range(retries):
        try:
            r = session.get(api, params=params, timeout=25)
            if r.status_code == 429:
                time.sleep(1.5 * (attempt + 1))
                continue
            r.raise_for_status()
            pages = r.json().get("query", {}).get("pages", {}) or {}
            for _, page in pages.items():
                qid = (page.get("pageprops") or {}).get("wikibase_item")
                if isinstance(qid, str) and qid.startswith("Q"):
                    return qid
            return None
        except Exception:
            time.sleep(0.5 * (attempt + 1))
    return None


def wikidata_search_qid(label: str, lang: str = "en", retries: int = 4) -> str | None:
    if not label or pd.isna(label):
        return None
    api = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbsearchentities",
        "format": "json",
        "language": lang or "en",
        "uselang": "en",
        "type": "item",
        "limit": 10,
        "search": label,
    }
    for attempt in range(retries):
        try:
            r = session.get(api, params=params, timeout=25)
            if r.status_code == 429:
                time.sleep(1.5 * (attempt + 1))
                continue
            r.raise_for_status()
            results = r.json().get("search", []) or []
            if not results:
                return None

            # Prefer things explicitly described as cathedrals/churches.
            for item in results:
                blob = f"{item.get('label','')} {item.get('description','')}".casefold()
                if any(k in blob for k in ["cathedral", "cathédrale", "cattedrale", "dom", "duomo", "church"]):
                    qid = item.get("id")
                    if isinstance(qid, str) and qid.startswith("Q"):
                        return qid

            qid = results[0].get("id")
            if isinstance(qid, str) and qid.startswith("Q"):
                return qid
            return None
        except Exception:
            time.sleep(0.5 * (attempt + 1))
    return None


def repair_missing_qids(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    wiki_col = first_existing_col(df, ["osmwikipedia_url_norm", "osmwikipedia", "wikipedia_url", "wikipedia"])
    if wiki_col is None:
        return df

    if "wikidata_qid" not in df.columns:
        df["wikidata_qid"] = np.nan

    missing = df["wikidata_qid"].isna() & df[wiki_col].notna()
    print(f"Repairing missing QIDs: {missing.sum()} candidate rows")

    for idx in tqdm(df.index[missing], desc="QID repair"):
        url = df.at[idx, wiki_col]
        qid = fetch_qid_from_wikipedia_url(url)

        if not qid:
            title = extract_wikipedia_title(url) or str(df.at[idx, "cathedral_name"])
            lang = wikipedia_language(url)
            qid = wikidata_search_qid(title, lang=lang)

        # English fallback on cathedral_name + city.
        if not qid:
            query = f"{df.at[idx, 'cathedral_name']} {df.at[idx, 'city']}"
            qid = wikidata_search_qid(query, lang="en")

        if qid:
            df.at[idx, "wikidata_qid"] = qid

        time.sleep(0.15)

    print("QIDs after repair:", df["wikidata_qid"].notna().sum(), "/", len(df))
    return df

# ------------------------------------------------------------
# 4) Re-query Wikidata style/material/height/surface for repaired QIDs
# ------------------------------------------------------------
WDQS_URL = "https://query.wikidata.org/sparql"


def chunks(xs: list[str], n: int):
    for i in range(0, len(xs), n):
        yield xs[i:i + n]


def wdqs_query(sparql: str, sleep_s: float = 1.0) -> list[dict[str, Any]]:
    try:
        r = session.get(
            WDQS_URL,
            params={"query": sparql},
            headers={"Accept": "application/sparql-results+json"},
            timeout=60,
        )
        if r.status_code == 429:
            time.sleep(4.0)
            return []
        r.raise_for_status()
        return r.json().get("results", {}).get("bindings", []) or []
    except Exception:
        return []
    finally:
        time.sleep(sleep_s)


def get_wikidata_labels(qids: list[str], prop: str, label_name: str) -> dict[str, str]:
    out: dict[str, list[str]] = {}
    qids = [q for q in qids if isinstance(q, str) and q.startswith("Q")]
    for batch in chunks(qids, 80):
        values = " ".join(f"wd:{q}" for q in batch)
        sparql = f"""
        SELECT ?item ?label WHERE {{
          VALUES ?item {{ {values} }}
          ?item wdt:{prop} ?thing .
          ?thing rdfs:label ?label .
          FILTER(LANG(?label) = "en")
        }}
        """
        for b in wdqs_query(sparql):
            qid = str(b.get("item", {}).get("value", "")).rsplit("/", 1)[-1]
            label = b.get("label", {}).get("value")
            if qid and label:
                out.setdefault(qid, []).append(label)

    return {qid: ", ".join(dict.fromkeys(vals)) for qid, vals in out.items()}


def get_wikidata_quantity(qids: list[str], prop: str) -> dict[str, float]:
    # Pull numeric quantity values only. WDQS normalizes many common quantity values.
    out: dict[str, list[float]] = {}
    qids = [q for q in qids if isinstance(q, str) and q.startswith("Q")]
    for batch in chunks(qids, 80):
        values = " ".join(f"wd:{q}" for q in batch)
        sparql = f"""
        SELECT ?item ?value WHERE {{
          VALUES ?item {{ {values} }}
          ?item wdt:{prop} ?value .
        }}
        """
        for b in wdqs_query(sparql):
            qid = str(b.get("item", {}).get("value", "")).rsplit("/", 1)[-1]
            val = b.get("value", {}).get("value")
            try:
                val_f = float(val)
            except Exception:
                continue
            if math.isfinite(val_f):
                out.setdefault(qid, []).append(val_f)

    # Use max because cathedrals may have multiple height statements for towers/spires.
    return {qid: max(vals) for qid, vals in out.items() if vals}


STYLE_PERIODS = {
    "romanesque": "c. 1000–1200",
    "gothic": "c. 1140–1600",
    "early gothic": "c. 1140–1200",
    "high gothic": "c. 1200–1300",
    "decorated gothic": "c. 1290–1350",
    "english gothic": "c. 1180–1520",
    "italian gothic": "c. 1140–1600",
    "baroque": "c. 1600–1750",
    "renaissance": "c. 1400–1600",
    "neoclassical": "c. 1750–1850",
    "byzantine": "c. 330–1453",
    "norman": "c. 1066–1200",
    "modernist": "c. 1900–present",
}


def style_with_periods(style_text: Any) -> str | float:
    if pd.isna(style_text) or not str(style_text).strip():
        return np.nan
    styles = [s.strip() for s in str(style_text).split(",") if s.strip()]
    out = []
    for s in styles:
        low = s.casefold()
        period = None
        # More specific keys first.
        for key in sorted(STYLE_PERIODS, key=len, reverse=True):
            if key in low:
                period = STYLE_PERIODS[key]
                break
        out.append(f"{s} ({period})" if period else s)
    return ", ".join(out)


def refresh_repaired_wikidata_fields(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "wikidata_qid" not in df.columns:
        return df

    qids = sorted({q for q in df["wikidata_qid"].dropna().astype(str) if q.startswith("Q")})
    print("Refreshing Wikidata P149/P186/P2048/P2046 for", len(qids), "QIDs")

    styles = get_wikidata_labels(qids, prop="P149", label_name="styleLabel")
    materials = get_wikidata_labels(qids, prop="P186", label_name="materialLabel")
    heights = get_wikidata_quantity(qids, prop="P2048")
    areas = get_wikidata_quantity(qids, prop="P2046")

    for idx, qid in df["wikidata_qid"].dropna().astype(str).items():
        if qid in styles:
            old = df.at[idx, "style_wikidata"] if "style_wikidata" in df.columns else np.nan
            # Fill missing style; preserve existing style unless it was missing.
            if pd.isna(old) or not str(old).strip():
                df.at[idx, "style_wikidata"] = styles[qid]
                df.at[idx, "style_source"] = "wikidata_p149"

        if qid in materials:
            if "material_final" not in df.columns or pd.isna(df.at[idx, "material_final"]):
                df.at[idx, "material_final"] = materials[qid]

        if qid in heights:
            val = heights[qid]
            if 5 <= val <= 200:
                if "height_m_wikidata" in df.columns:
                    df.at[idx, "height_m_wikidata"] = val
                if "height_m" in df.columns and pd.isna(pd.to_numeric(df.at[idx, "height_m"], errors="coerce")):
                    df.at[idx, "height_m"] = val
                    df.at[idx, "height_source"] = "wikidata_p2048_filled_missing"

        if qid in areas:
            val = areas[qid]
            if 20 <= val <= 500000:
                if "surface_m2_wikidata" in df.columns:
                    df.at[idx, "surface_m2_wikidata"] = val
                if "surface_m2" in df.columns and pd.isna(pd.to_numeric(df.at[idx, "surface_m2"], errors="coerce")):
                    df.at[idx, "surface_m2"] = val
                    df.at[idx, "surface_source"] = "wikidata_p2046_filled_missing"

    df["style_wikidata_with_periods"] = df["style_wikidata"].apply(style_with_periods)
    return df

# ------------------------------------------------------------
# 5) Targeted source-checked corrections for rows known to drift
# ------------------------------------------------------------
TARGETED_RESEARCH_CORRECTIONS = [
    {
        "name": "St. Petri-Dom", "city": "Bremen", "country": "de",
        "updates": {"height_m": 98.0, "height_m_wikidata": 98.0, "height_source": "targeted_source_checked"},
        "source": "SkyDB / Bremen tourism: St. Petri Dom twin spires around 98m",
    },
    {
        "name": "Dom", "city": "Paderborn", "country": "de",
        "updates": {"height_m": 93.0, "height_m_wikidata": 93.0, "height_source": "targeted_source_checked"},
        "source": "Paderborn Cathedral sources: tower 93m",
    },
    {
        "name": "Cathédrale Sainte-Cécile d'Albi", "city": "Albi", "country": "fr",
        # Current major web/Wikipedia-style sources list the total height as 78m.
        # The old 40m value likely referred to a narrower architectural sub-measure.
        "updates": {"height_m": 78.0, "height_m_wikidata": 78.0, "height_source": "targeted_source_checked"},
        "source": "Albi Cathedral infobox/current web data: height 78m",
    },
    {
        "name": "Duomo di Monza", "city": "Monza", "country": "it",
        "updates": {
            "end_year": 1681,
            "end_year_wikipedia_infobox": 1681,
            "end_year_wikipedia_final": 1681,
            "end_year_source": "targeted_source_checked",
            "wikipedia_date_notes": np.nan,
        },
        "source": "Duomo of Monza infobox: year completed 1681",
    },
    {
        "name": "Cattedrale di Sant'Agata", "city": "Catania", "country": "it",
        "updates": {
            "start_year": 1086,
            "start_year_wikipedia_earliest": 1086,
            "start_year_wikipedia_final": 1086,
            "start_year_source": "targeted_source_checked",
            "wikipedia_date_notes": "earliest: risale al periodo 1086-1094; targeted source-checked",
        },
        "source": "Catania Cathedral sources: first building/construction in the 1086-1094 range",
    },
    {
        "name": "Sankt-Petri-Dom zu Schleswig", "city": "Schleswig (hedeby)", "country": "de",
        "updates": {"height_m": 112.0, "height_m_wikidata": 112.0, "height_source": "targeted_source_checked"},
        "source": "Schleswig Cathedral sources: central/western tower 112m",
    },
    {
        "name": "Dom Mariä Himmelfahrt", "city": "Hildesheim", "country": "de",
        "updates": {"surface_m2": 5800.0, "surface_m2_wikidata": 5800.0, "surface_source": "targeted_source_checked"},
        "source": "Hildesheim Cathedral UNESCO/Wikipedia area: 0.58ha = 5800m²",
    },
]


def apply_targeted_corrections(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = df.copy()
    audit_rows = []

    for item in TARGETED_RESEARCH_CORRECTIONS:
        mask = row_mask(df, item["name"], item.get("city"), item.get("country"))
        n = int(mask.sum())
        if n == 0:
            audit_rows.append({
                "target": item["name"],
                "city": item.get("city"),
                "country": item.get("country"),
                "status": "not_found",
                "column": None,
                "old_value": None,
                "new_value": None,
                "source": item["source"],
            })
            continue
        if n > 1:
            audit_rows.append({
                "target": item["name"],
                "city": item.get("city"),
                "country": item.get("country"),
                "status": f"multiple_matches_{n}",
                "column": None,
                "old_value": None,
                "new_value": None,
                "source": item["source"],
            })
            continue

        idx = df.index[mask][0]
        for col, new_val in item["updates"].items():
            if col not in df.columns:
                df[col] = np.nan
            old_val = df.at[idx, col]
            df.at[idx, col] = new_val
            audit_rows.append({
                "target": item["name"],
                "city": item.get("city"),
                "country": item.get("country"),
                "status": "updated",
                "row_index": int(idx),
                "column": col,
                "old_value": old_val,
                "new_value": new_val,
                "source": item["source"],
            })

    return df, pd.DataFrame(audit_rows)

# ------------------------------------------------------------
# 6) Elevation source: Open-Elevation API
# ------------------------------------------------------------
def load_elevation_cache() -> dict[tuple[float, float], float]:
    cache = {}
    if ELEVATION_CACHE_CSV.exists():
        c = pd.read_csv(ELEVATION_CACHE_CSV)
        for _, r in c.iterrows():
            lat = pd.to_numeric(r.get("lat"), errors="coerce")
            lon = pd.to_numeric(r.get("lon"), errors="coerce")
            elev = pd.to_numeric(r.get("elevation_m"), errors="coerce")
            if pd.notna(lat) and pd.notna(lon) and pd.notna(elev):
                cache[(round(float(lat), 6), round(float(lon), 6))] = float(elev)
    return cache


def save_elevation_cache(cache: dict[tuple[float, float], float]) -> None:
    rows = [{"lat": k[0], "lon": k[1], "elevation_m": v} for k, v in cache.items()]
    pd.DataFrame(rows).to_csv(ELEVATION_CACHE_CSV, index=False)


def fetch_open_elevation_batch(points: list[tuple[float, float]]) -> list[float | None]:
    url = "https://api.open-elevation.com/api/v1/lookup"
    payload = {"locations": [{"latitude": lat, "longitude": lon} for lat, lon in points]}
    try:
        r = session.post(url, json=payload, timeout=45)
        r.raise_for_status()
        data = r.json()
        results = data.get("results", []) or []
        vals = []
        for res in results:
            elev = res.get("elevation")
            vals.append(float(elev) if elev is not None else None)
        if len(vals) == len(points):
            return vals
    except Exception:
        pass
    return [None] * len(points)


def add_elevation(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "elevation_m" not in df.columns:
        df["elevation_m"] = np.nan

    if "lat" not in df.columns or "lon" not in df.columns:
        return df

    cache = load_elevation_cache()
    missing_idx = df.index[df["elevation_m"].isna() & df["lat"].notna() & df["lon"].notna()].tolist()
    print("Elevation missing rows:", len(missing_idx))

    # Fill from cache first.
    still_missing = []
    for idx in missing_idx:
        lat = float(pd.to_numeric(df.at[idx, "lat"], errors="coerce"))
        lon = float(pd.to_numeric(df.at[idx, "lon"], errors="coerce"))
        key = (round(lat, 6), round(lon, 6))
        if key in cache:
            df.at[idx, "elevation_m"] = cache[key]
        else:
            still_missing.append(idx)

    # Query API in batches.
    for batch_idx in tqdm(list(chunks(still_missing, 75)), desc="Open-Elevation"):
        points = []
        keys = []
        for idx in batch_idx:
            lat = float(pd.to_numeric(df.at[idx, "lat"], errors="coerce"))
            lon = float(pd.to_numeric(df.at[idx, "lon"], errors="coerce"))
            key = (round(lat, 6), round(lon, 6))
            points.append((lat, lon))
            keys.append((idx, key))

        elevations = fetch_open_elevation_batch(points)
        for (idx, key), elev in zip(keys, elevations):
            if elev is not None and math.isfinite(elev):
                df.at[idx, "elevation_m"] = elev
                cache[key] = elev
        time.sleep(1.0)

    save_elevation_cache(cache)
    return df

# ------------------------------------------------------------
# 7) Final cleanup
# ------------------------------------------------------------
def ensure_original_columns_only(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "study_id" not in df.columns:
        df.insert(0, "study_id", np.arange(1, len(df) + 1))

    # Keep old notebook style: numeric fields as numeric where possible.
    numeric_cols = [
        "study_id", "lat", "lon", "start_year", "end_year", "duration_years",
        "surface_m2", "height_m", "volume_proxy", "n_disasters",
        "first_disaster_year", "last_disaster_year", "disasters_during_construction",
        "war_during_construction", "fire_during_construction", "earthquake_during_construction",
        "start_year_wikipedia_infobox", "end_year_wikipedia_infobox",
        "start_year_wikipedia_earliest", "start_year_wikipedia_current",
        "end_year_wikipedia_current", "start_year_wikipedia_final", "end_year_wikipedia_final",
        "start_year_original", "end_year_original", "height_m_wikidata",
        "surface_m2_wikidata", "elevation_m",
    ]
    for c in numeric_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    if RECOMPUTE_DURATION_AFTER_TARGETED_DATE_FIXES:
        df["duration_years"] = df["end_year"] - df["start_year"]
        df["period_bucket"] = df["start_year"].apply(period_bucket)

    # Recompute volume only when both components are present.
    if "surface_m2" in df.columns and "height_m" in df.columns:
        surface = pd.to_numeric(df["surface_m2"], errors="coerce")
        height = pd.to_numeric(df["height_m"], errors="coerce")
        if "volume_proxy" not in df.columns:
            df["volume_proxy"] = np.nan
        missing_or_bad_volume = df["volume_proxy"].isna() & surface.notna() & height.notna()
        df.loc[missing_or_bad_volume, "volume_proxy"] = surface[missing_or_bad_volume] * height[missing_or_bad_volume]

    # Create any missing original columns as blank so final order is exact.
    for c in ORIGINAL_FINAL_COLUMNS:
        if c not in df.columns:
            df[c] = np.nan

    return df[ORIGINAL_FINAL_COLUMNS]

# ------------------------------------------------------------
# 8) Run
# ------------------------------------------------------------
def main() -> None:
    if not INPUT_CSV.exists():
        raise FileNotFoundError(f"Could not find input CSV: {INPUT_CSV}")

    print("Reading:", INPUT_CSV)
    df = pd.read_csv(INPUT_CSV)
    print("Input shape:", df.shape)

    df = repair_missing_qids(df)
    df = refresh_repaired_wikidata_fields(df)
    df, audit = apply_targeted_corrections(df)
    df = add_elevation(df)
    final = ensure_original_columns_only(df)

    final.to_csv(OUTPUT_CSV, index=False)
    audit.to_csv(AUDIT_CSV, index=False)

    print("\nSaved final original-column file:", OUTPUT_CSV)
    print("Final shape:", final.shape)
    print("Saved audit file:", AUDIT_CSV)
    print("Columns:")
    for c in final.columns:
        print(" -", c)


if __name__ == "__main__":
    main()


Reading: /Users/jeeveshattri/Desktop/Monuments_Of_Power/Background_Data/cathedral_master_with_wikidata_styles_with_periods.csv
Input shape: (313, 66)
Repairing missing QIDs: 39 candidate rows


QID repair: 100%|██████████| 39/39 [02:15<00:00,  3.47s/it]


QIDs after repair: 308 / 313
Refreshing Wikidata P149/P186/P2048/P2046 for 308 QIDs
Elevation missing rows: 313


Open-Elevation: 100%|██████████| 5/5 [00:08<00:00,  1.61s/it]


Saved final original-column file: /Users/jeeveshattri/Desktop/Monuments_Of_Power/Background_Data/cathedral_final_researched_clean.csv
Final shape: (313, 56)
Saved audit file: /Users/jeeveshattri/Desktop/Monuments_Of_Power/Background_Data/cathedral_targeted_research_audit.csv
Columns:
 - study_id
 - cathedral_name
 - city
 - country
 - lat
 - lon
 - start_year
 - end_year
 - duration_years
 - period_bucket
 - surface_m2
 - height_m
 - volume_proxy
 - completed
 - not_collapsed
 - n_disasters
 - any_war
 - any_fire
 - any_earthquake
 - any_other_natural
 - any_other
 - first_disaster_year
 - last_disaster_year
 - disasters_during_construction
 - war_during_construction
 - fire_during_construction
 - earthquake_during_construction
 - disaster_years
 - saints
 - destroyed
 - style_wikidata
 - style_source
 - material_final
 - start_year_wikipedia_infobox
 - end_year_wikipedia_infobox
 - start_year_wikipedia_infobox_label
 - end_year_wikipedia_infobox_label
 - start_year_wikipedia_earliest

In [4]:
# ============================================================
# Cathedral OSM Area Enrichment Cell - stricter safe version
# Adds OSM polygon/footprint area where cathedral surface_m2 is missing
#
# Important behavior:
#   1. Runs ONE test query first.
#   2. If the first query does not return a trustworthy OSM area, the script STOPS.
#   3. If the first query works, it proceeds through the remaining rows.
#   4. Avoids false matches like Johanniskirche -> Mainz Cathedral.
#   5. Fixes pandas dtype warnings for text columns.
# ============================================================

from pathlib import Path
import time
import math
import json
import requests
import numpy as np
import pandas as pd

try:
    from pyproj import Geod
    HAS_GEOM = True
except Exception:
    HAS_GEOM = False
    print("WARNING: pyproj not available. OSM polygon area cannot be calculated.")


# -----------------------------
# Config
# -----------------------------

INPUT_CATHEDRAL_FILE = Path("/Users/jeeveshattri/Desktop/Monuments_Of_Power/Background_Data/cathedral_final_researched_clean.csv")
OUTPUT_CATHEDRAL_FILE = Path("/Users/jeeveshattri/Desktop/Monuments_Of_Power/Background_Data/cathedral_final_researched_clean_osm_area.csv")
OSM_CACHE_FILE = Path("/Users/jeeveshattri/Desktop/Monuments_Of_Power/Background_Data/cathedral_osm_area_cache.json")

OVERPASS_URL = "https://overpass-api.de/api/interpreter"

SEARCH_RADIUS_M = 400
SLEEP_SECONDS = 1.2
ONLY_FILL_MISSING_SURFACE = True

# If True, deletes old cache at start. Keep True while testing.
DELETE_OLD_CACHE = True

GEOD = Geod(ellps="WGS84") if HAS_GEOM else None


# -----------------------------
# Optional: delete failed/old cache
# -----------------------------

if DELETE_OLD_CACHE and OSM_CACHE_FILE.exists():
    OSM_CACHE_FILE.unlink()
    print(f"Deleted old OSM cache: {OSM_CACHE_FILE}")


# -----------------------------
# Helpers
# -----------------------------

def normalize_text(x):
    if pd.isna(x):
        return ""
    return str(x).lower().strip()


def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    )
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def polygon_area_m2_from_coords(coords):
    """
    Compute geodesic polygon area from [(lat, lon), ...].
    """
    if not HAS_GEOM or not coords or len(coords) < 4:
        return np.nan

    lats = [p[0] for p in coords]
    lons = [p[1] for p in coords]

    try:
        area, _ = GEOD.polygon_area_perimeter(lons, lats)
        return abs(area)
    except Exception:
        return np.nan


def centroid_from_coords(coords):
    if not coords:
        return (np.nan, np.nan)
    return (
        float(np.mean([p[0] for p in coords])),
        float(np.mean([p[1] for p in coords]))
    )


def name_match_score(cathedral_name, osm_name):
    """
    Lightweight token-overlap score.
    Higher means stronger name similarity.
    """
    a = normalize_text(cathedral_name)
    b = normalize_text(osm_name)

    if not a or not b:
        return 0.0

    generic = {
        "cathedral", "cathédrale", "catedral", "cattedrale",
        "duomo", "dom", "domkirche", "kathedrale", "katedra",
        "church", "basilica", "basilique", "minster",
        "st", "saint", "santa", "san", "sankt", "sainte",
        "de", "di", "of", "the", "la", "le", "il", "et",
        "and", "und", "zu", "del", "della", "dei", "des"
    }

    def tokens(s):
        cleaned = (
            s.replace(",", " ")
             .replace(".", " ")
             .replace("-", " ")
             .replace("'", " ")
             .replace('"', " ")
             .replace("(", " ")
             .replace(")", " ")
             .replace("/", " ")
        )
        return {
            token for token in cleaned.split()
            if token not in generic and len(token) > 2
        }

    ta = tokens(a)
    tb = tokens(b)

    if not ta or not tb:
        return 0.0

    return len(ta & tb) / len(ta | tb)


def get_all_osm_names(tags):
    """
    Combine all useful OSM name fields.
    This helps because OSM may store English/French/Italian/German names separately.
    """
    if not isinstance(tags, dict):
        return ""

    name_keys = [
        "name", "alt_name", "official_name", "short_name",
        "name:en", "name:fr", "name:it", "name:de",
        "name:es", "name:la", "old_name"
    ]

    values = []
    for key in name_keys:
        value = tags.get(key)
        if value:
            values.append(str(value))

    return " ; ".join(values)


def tags_look_like_cathedral(tags, cathedral_name=""):
    """
    Candidate pre-filter.
    Still permissive, but candidate ranking later is stricter.
    """
    if not isinstance(tags, dict):
        return False

    vals = {str(k).lower(): str(v).lower() for k, v in tags.items()}

    building = vals.get("building", "")
    amenity = vals.get("amenity", "")
    historic = vals.get("historic", "")
    religion = vals.get("religion", "")
    denomination = vals.get("denomination", "")
    tourism = vals.get("tourism", "")

    all_names = get_all_osm_names(tags).lower()

    cathedral_words = [
        "cathedral", "cathédrale", "catedral", "cattedrale",
        "duomo", "dom", "domkirche", "kathedrale", "katedra",
        "minster", "basilica", "basilique"
    ]

    name_says_cathedral = any(w in all_names for w in cathedral_words)

    religious_tag = (
        amenity == "place_of_worship"
        or religion == "christian"
        or denomination != ""
        or building in {"cathedral", "church", "chapel", "basilica", "religious", "yes"}
        or historic in {"cathedral", "church", "monastery", "religious"}
        or tourism == "attraction"
    )

    nscore = name_match_score(cathedral_name, all_names)

    return religious_tag or name_says_cathedral or nscore > 0.15


def build_overpass_query(lat, lon, radius_m):
    """
    Corrected Overpass query.

    The earlier broken version used:
        out geom tags center;

    This version uses:
        out body geom;
    """
    return f"""
[out:json][timeout:90];
(
  way(around:{radius_m},{lat},{lon})["building"];
  relation(around:{radius_m},{lat},{lon})["building"];

  way(around:{radius_m},{lat},{lon})["amenity"="place_of_worship"];
  relation(around:{radius_m},{lat},{lon})["amenity"="place_of_worship"];

  way(around:{radius_m},{lat},{lon})["historic"];
  relation(around:{radius_m},{lat},{lon})["historic"];

  way(around:{radius_m},{lat},{lon})["name"];
  relation(around:{radius_m},{lat},{lon})["name"];
);
out body geom;
"""


def overpass_search_cathedral_area(lat, lon, cathedral_name, radius_m=400):
    query = build_overpass_query(lat, lon, radius_m)

    try:
        response = requests.post(
            OVERPASS_URL,
            data={"data": query},
            headers={"User-Agent": "cathedral-osm-area-enrichment/1.0"},
            timeout=120
        )

        if response.status_code != 200:
            return {
                "osm_area_m2": np.nan,
                "osm_area_confidence": "none",
                "osm_match_name": None,
                "osm_match_type": None,
                "osm_match_id": np.nan,
                "osm_match_distance_m": np.nan,
                "osm_match_score": np.nan,
                "osm_name_score": np.nan,
                "osm_match_tags": None,
                "osm_area_note": f"overpass_error_{response.status_code}: {response.text[:700]}",
            }

        data = response.json()

    except Exception as e:
        return {
            "osm_area_m2": np.nan,
            "osm_area_confidence": "none",
            "osm_match_name": None,
            "osm_match_type": None,
            "osm_match_id": np.nan,
            "osm_match_distance_m": np.nan,
            "osm_match_score": np.nan,
            "osm_name_score": np.nan,
            "osm_match_tags": None,
            "osm_area_note": f"overpass_exception: {e}",
        }

    candidates = []

    for el in data.get("elements", []):
        tags = el.get("tags", {}) or {}

        if not tags_look_like_cathedral(tags, cathedral_name):
            continue

        geom = el.get("geometry", [])
        coords = [(p["lat"], p["lon"]) for p in geom if "lat" in p and "lon" in p]

        if len(coords) < 4:
            continue

        first = coords[0]
        last = coords[-1]

        # Require basically closed polygon geometry.
        if haversine_m(first[0], first[1], last[0], last[1]) > 10:
            continue

        area_m2 = polygon_area_m2_from_coords(coords)

        if pd.isna(area_m2) or area_m2 <= 0:
            continue

        # Broad but sane bounds for cathedral footprint or precinct.
        if area_m2 < 100 or area_m2 > 800000:
            continue

        c_lat, c_lon = centroid_from_coords(coords)
        dist_m = haversine_m(lat, lon, c_lat, c_lon)

        all_names = get_all_osm_names(tags)
        osm_name = tags.get("name", "") or all_names
        nscore = name_match_score(cathedral_name, all_names)

        building = str(tags.get("building", "")).lower()
        amenity = str(tags.get("amenity", "")).lower()
        historic = str(tags.get("historic", "")).lower()
        religion = str(tags.get("religion", "")).lower()

        explicit_bonus = 0

        if building == "cathedral":
            explicit_bonus += 4
        elif building in {"church", "basilica", "chapel"}:
            explicit_bonus += 2
        elif building == "yes":
            explicit_bonus += 0.5

        if amenity == "place_of_worship":
            explicit_bonus += 2

        if historic in {"cathedral", "church"}:
            explicit_bonus += 2

        if religion == "christian":
            explicit_bonus += 1

        distance_score = max(0, 1 - (dist_m / radius_m))

        # Stronger name weight than before.
        # Also penalize bad name matches that are not very close.
        name_mismatch_penalty = 0
        if nscore < 0.15 and dist_m > 60:
            name_mismatch_penalty = 5

        total_score = (
            4.0 * distance_score
            + 7.0 * nscore
            + explicit_bonus
            - name_mismatch_penalty
        )

        # Hard rejection:
        # If name does not match and the candidate is not extremely close,
        # do not accept it just because it is a big nearby cathedral.
        if nscore < 0.15 and dist_m > 120:
            continue

        candidates.append({
            "osm_area_m2": area_m2,
            "osm_match_name": osm_name if osm_name else None,
            "osm_match_type": el.get("type"),
            "osm_match_id": el.get("id"),
            "osm_match_distance_m": dist_m,
            "osm_match_score": total_score,
            "osm_name_score": nscore,
            "osm_match_tags": json.dumps(tags, ensure_ascii=False),
            "osm_area_note": "osm_polygon_area_used_as_mapped_footprint_or_precinct_estimate",
        })

    if not candidates:
        return {
            "osm_area_m2": np.nan,
            "osm_area_confidence": "none",
            "osm_match_name": None,
            "osm_match_type": None,
            "osm_match_id": np.nan,
            "osm_match_distance_m": np.nan,
            "osm_match_score": np.nan,
            "osm_name_score": np.nan,
            "osm_match_tags": None,
            "osm_area_note": "no_suitable_osm_polygon_found",
        }

    best = sorted(
        candidates,
        key=lambda x: (
            x["osm_match_score"],
            -x["osm_match_distance_m"],
            x["osm_area_m2"]
        ),
        reverse=True
    )[0]

    dist = best["osm_match_distance_m"]
    score = best["osm_match_score"]
    nscore = best["osm_name_score"]

    # Stricter confidence:
    # High = close or name-matched.
    # Medium = acceptable but not perfect.
    # Low = usable but should be reviewed.
    if (dist <= 80 and score >= 4 and nscore >= 0.15) or dist <= 35:
        confidence = "high"
    elif (dist <= 150 and score >= 2.5 and nscore >= 0.10) or dist <= 60:
        confidence = "medium"
    else:
        confidence = "low"

    best["osm_area_confidence"] = confidence
    return best


def validate_first_query_result(first_result, first_name):
    """
    Decide whether the first query is safe enough to allow the full run.
    This prevents the script from continuing after a suspicious first match.
    """
    first_area = pd.to_numeric(first_result.get("osm_area_m2"), errors="coerce")
    first_conf = first_result.get("osm_area_confidence")
    first_match_name = str(first_result.get("osm_match_name") or "")
    first_dist = pd.to_numeric(first_result.get("osm_match_distance_m"), errors="coerce")
    first_name_score = pd.to_numeric(first_result.get("osm_name_score"), errors="coerce")

    if pd.isna(first_name_score):
        first_name_score = name_match_score(first_name, first_match_name)

    print("\nFirst query validation:")
    print(f"Area: {first_area}")
    print(f"Confidence: {first_conf}")
    print(f"Matched OSM name: {first_match_name}")
    print(f"Distance m: {first_dist}")
    print(f"Name match score: {first_name_score}")

    first_query_ok = (
        pd.notna(first_area)
        and first_area > 0
        and first_conf in {"high", "medium"}
        and pd.notna(first_dist)
        and (
            first_name_score >= 0.20
            or first_dist <= 60
        )
    )

    return first_query_ok, {
        "first_area": first_area,
        "first_conf": first_conf,
        "first_match_name": first_match_name,
        "first_dist": first_dist,
        "first_name_score": first_name_score,
    }


def make_json_safe(obj):
    """
    Prevents invalid/nonstandard JSON cache values.
    Converts NaN/inf to None.
    """
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [make_json_safe(v) for v in obj]
    if isinstance(obj, float):
        if math.isnan(obj) or math.isinf(obj):
            return None
    return obj


def load_cache(path):
    if path.exists():
        try:
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            return {}
    return {}


def save_cache(cache, path):
    safe_cache = make_json_safe(cache)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(safe_cache, f, ensure_ascii=False, indent=2)


# -----------------------------
# Main enrichment
# -----------------------------

if not HAS_GEOM:
    raise RuntimeError("pyproj is required for polygon area calculation. Install pyproj before running.")

df = pd.read_csv(INPUT_CATHEDRAL_FILE)

required_cols = ["cathedral_name", "lat", "lon", "surface_m2"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns in cathedral file: {missing}")

df["surface_m2"] = pd.to_numeric(df["surface_m2"], errors="coerce")
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")

# Stable OSM column dtypes: prevents pandas FutureWarning
numeric_osm_cols = [
    "osm_area_m2",
    "osm_match_id",
    "osm_match_distance_m",
    "osm_match_score",
    "osm_name_score",
]

text_osm_cols = [
    "osm_area_confidence",
    "osm_match_name",
    "osm_match_type",
    "osm_match_tags",
    "osm_area_note",
]

for col in numeric_osm_cols:
    if col not in df.columns:
        df[col] = np.nan
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in text_osm_cols:
    if col not in df.columns:
        df[col] = None
    df[col] = df[col].astype("object")

osm_cols_defaults = {
    "osm_area_m2": np.nan,
    "osm_area_confidence": None,
    "osm_match_name": None,
    "osm_match_type": None,
    "osm_match_id": np.nan,
    "osm_match_distance_m": np.nan,
    "osm_match_score": np.nan,
    "osm_name_score": np.nan,
    "osm_match_tags": None,
    "osm_area_note": None,
}

if ONLY_FILL_MISSING_SURFACE:
    query_mask = df["surface_m2"].isna() | (df["surface_m2"] <= 0)
else:
    query_mask = pd.Series(True, index=df.index)

query_mask = query_mask & df["lat"].notna() & df["lon"].notna()
rows_to_query = df[query_mask].copy()

print(f"Total cathedrals: {len(df)}")
print(f"Rows with existing surface_m2: {(df['surface_m2'].notna() & (df['surface_m2'] > 0)).sum()}")
print(f"Rows selected for OSM area search: {len(rows_to_query)}")

if len(rows_to_query) == 0:
    print("No rows need OSM enrichment.")
else:
    # --------------------------------------------------------
    # First-query gate
    # --------------------------------------------------------
    first_i = rows_to_query.index[0]
    first_row = rows_to_query.loc[first_i]

    first_name = str(first_row["cathedral_name"])
    first_lat = float(first_row["lat"])
    first_lon = float(first_row["lon"])

    print("\nTesting first OSM query before full run:")
    print(f"Index: {first_i}")
    print(f"Name: {first_name}")
    print(f"Lat/Lon: {first_lat}, {first_lon}")

    first_result = overpass_search_cathedral_area(
        lat=first_lat,
        lon=first_lon,
        cathedral_name=first_name,
        radius_m=SEARCH_RADIUS_M
    )

    print("\nFirst query result:")
    print(first_result)

    first_query_ok, first_debug = validate_first_query_result(first_result, first_name)

    if not first_query_ok:
        raise RuntimeError(
            "\nSTOPPING: First OSM query did not return a trustworthy area. "
            "The full loop was not run.\n"
            f"Input cathedral name: {first_name}\n"
            f"OSM matched name: {first_debug['first_match_name']}\n"
            f"Area: {first_debug['first_area']}\n"
            f"Confidence: {first_debug['first_conf']}\n"
            f"Distance: {first_debug['first_dist']} m\n"
            f"Name match score: {first_debug['first_name_score']}\n"
            f"Reason/note: {first_result.get('osm_area_note')}"
        )

    print("\nFirst query worked and passed validation. Proceeding with full OSM enrichment...\n")

    cache = load_cache(OSM_CACHE_FILE)

    for i, row in rows_to_query.iterrows():
        name = str(row["cathedral_name"])
        lat = float(row["lat"])
        lon = float(row["lon"])

        cache_key = f"{name}|{lat:.6f}|{lon:.6f}|r{SEARCH_RADIUS_M}"

        if i == first_i:
            result = first_result
        elif cache_key in cache:
            result = cache[cache_key]
        else:
            print(f"OSM search: {i} | {name}")
            result = overpass_search_cathedral_area(
                lat=lat,
                lon=lon,
                cathedral_name=name,
                radius_m=SEARCH_RADIUS_M
            )
            cache[cache_key] = result
            save_cache(cache, OSM_CACHE_FILE)
            time.sleep(SLEEP_SECONDS)

        for col in osm_cols_defaults:
            df.at[i, col] = result.get(col, osm_cols_defaults[col])

    first_cache_key = f"{first_name}|{first_lat:.6f}|{first_lon:.6f}|r{SEARCH_RADIUS_M}"
    cache[first_cache_key] = first_result
    save_cache(cache, OSM_CACHE_FILE)


# -----------------------------
# Final area fields
# -----------------------------

df["area_m2_final"] = df["surface_m2"]
df["osm_area_m2"] = pd.to_numeric(df["osm_area_m2"], errors="coerce")

# I recommend only using high/medium in the first serious run.
# You can include low later if you manually inspect.
ACCEPTED_OSM_CONFIDENCE = {"high", "medium"}

use_osm_mask = (
    (df["area_m2_final"].isna() | (df["area_m2_final"] <= 0))
    & df["osm_area_m2"].notna()
    & (df["osm_area_m2"] > 0)
    & df["osm_area_confidence"].isin(ACCEPTED_OSM_CONFIDENCE)
)

df.loc[use_osm_mask, "area_m2_final"] = df.loc[use_osm_mask, "osm_area_m2"]

df["area_m2_source"] = np.where(
    df["surface_m2"].notna() & (df["surface_m2"] > 0),
    "original_surface_m2",
    np.where(use_osm_mask, "osm_polygon_area", "missing")
)

df["log_area_m2_final"] = np.where(
    df["area_m2_final"].notna() & (df["area_m2_final"] > 0),
    np.log(df["area_m2_final"]),
    np.nan
)

df["has_original_surface_m2"] = df["surface_m2"].notna() & (df["surface_m2"] > 0)
df["has_osm_area"] = df["osm_area_m2"].notna() & (df["osm_area_m2"] > 0)
df["has_area_m2_final"] = df["area_m2_final"].notna() & (df["area_m2_final"] > 0)

df.to_csv(OUTPUT_CATHEDRAL_FILE, index=False)

print("\nDone.")
print(f"Saved: {OUTPUT_CATHEDRAL_FILE}")

print("\nArea coverage summary:")
print(df["area_m2_source"].value_counts(dropna=False))

print("\nOSM confidence summary:")
print(df["osm_area_confidence"].value_counts(dropna=False))

print("\nFinal usable area rows:")
print(df["has_area_m2_final"].value_counts(dropna=False))

print("\nPotential review rows: OSM area found but not used because confidence was low/none")
review_mask = (
    df["surface_m2"].isna()
    & df["osm_area_m2"].notna()
    & (df["osm_area_m2"] > 0)
    & ~df["osm_area_confidence"].isin(ACCEPTED_OSM_CONFIDENCE)
)

review_cols = [
    "cathedral_name",
    "osm_area_m2",
    "osm_area_confidence",
    "osm_match_name",
    "osm_match_distance_m",
    "osm_name_score",
    "osm_area_note",
]

print(df.loc[review_mask, review_cols].head(25))

Total cathedrals: 313
Rows with existing surface_m2: 64
Rows selected for OSM area search: 249

Testing first OSM query before full run:
Index: 0
Name: Johanniskirche
Lat/Lon: 49.99841389, 8.272385862

First query result:
{'osm_area_m2': 1252.3721325574443, 'osm_match_name': 'Alter Dom Sankt Johannis', 'osm_match_type': 'way', 'osm_match_id': 100010435, 'osm_match_distance_m': 4.427925987697783, 'osm_match_score': 8.955720740123022, 'osm_name_score': 0.0, 'osm_match_tags': '{"addr:city": "Mainz", "addr:country": "DE", "addr:housenumber": "1", "addr:postcode": "55116", "addr:street": "Schöfferstraße", "alt_name": "Johanniskirche;Aldeduom;Alter Dom;Sankt-Johannis-Kirche", "alt_name:en": "Saint John\'s Church", "amenity": "place_of_worship", "building": "church", "building:colour": "white", "deanery": "Dekanat Mainz", "denomination": "protestant", "diocese": "Evangelische Kirche in Hessen und Nassau", "full_name": "Alter Dom Sankt Johannis", "heritage": "yes", "historic": "yes", "image": 